# Scalable GPS Route Clustering

**Pipeline:**
Raw GPS → cleaning → trip construction → segmentation → spatial partitioning (Grid/Quadtree/KDB) → cell sequences → OD grouping → candidate pairs → route similarity (Jaccard/LCSS/Edit) → frequent routes → similarity graph → label propagation → anomaly scoring → evaluation → scalability

## Dataset source

This project uses the Beijing taxi trajectory / T-Drive taxi GPS dataset.

Public dataset source:

A. Niculescu, “T-Drive trajectory data sample,” Kaggle.  
https://www.kaggle.com/datasets/arashnic/tdriver


## Configuration & Spark Setup

Initialises the Spark session, defines pipeline configuration parameters, runtime flags, and helper utilities for timing, display, and output saving.


In [ ]:
import os
import math
import time
from dataclasses import dataclass
from contextlib import contextmanager
from typing import List, Tuple, Dict, Optional

try:
    spark  
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("COMP4124_Taxi_Route_Clustering").getOrCreate()

from pyspark.sql import SparkSession, DataFrame, Window, Column
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from pyspark.storagelevel import StorageLevel

@dataclass(frozen=True)
class PipelineConfig:
    RAW_INPUT_PATH: str = "abfss://2026-group4@sauondbrwebigdatalrs1.dfs.core.windows.net/taxi_log_2008_by_id/"
    RAW_HAS_HEADER: bool = False
    RAW_SEPARATOR: str = ","

    # Geographic bounding box
    MIN_LON: float = 115.0
    MAX_LON: float = 117.5
    MIN_LAT: float = 39.0
    MAX_LAT: float = 41.0

    # Trip construction thresholds
    TRIP_GAP_MINUTES: int = 10
    MAX_POINT_JUMP_KM: float = 5.0
    MAX_SPEED_KMH: float = 160.0
    CELL_SIZE_DEG: float = 0.005

    # Spatial partitioning configs
    GRID_X: int = 10
    GRID_Y: int = 10
    QT_MAX_POINTS: int = 100_000
    QT_MAX_DEPTH: int = 5
    KDB_MAX_POINTS: int = 300_000
    KDB_MAX_DEPTH: int = 4

    # Similarity + clustering thresholds
    MIN_SHARED_CELLS: int = 2
    MAX_DISTANCE_RATIO_DIFF: float = 0.40
    MAX_DURATION_RATIO_DIFF: float = 0.60
    MIN_JACCARD_SIMILARITY: float = 0.20
    MIN_SEQUENCE_SIMILARITY: float = 0.30
    ROUTE_EDGE_SIMILARITY_THRESHOLD: float = 0.45
    RANDOM_SEED: int = 42

cfg = PipelineConfig()

# Runtime flags
DEMO_MODE = False
DEMO_LIMIT = 15_000
RUN_PARTITIONING_EVALUATION = True
RUN_SCALABILITY_BENCHMARK = True

WRITE_OUTPUTS = False
OUTPUT_ROOT = os.environ.get("TAXI_OUTPUT_ROOT", "/tmp/comp4124_taxi_outputs")

runtime_results: List[Tuple[str, float]] = []

def maybe_limit(df: DataFrame, n: int = DEMO_LIMIT) -> DataFrame:
    return df.limit(n) if DEMO_MODE else df

def display_df(df: DataFrame, n: int = 20, truncate: bool = False):
    try:
        display(df.limit(n))
    except Exception:
        df.show(n, truncate=truncate)

def output_path(name: str) -> str:
    return os.path.join(OUTPUT_ROOT, name)

def save_if_enabled(df: DataFrame, name: str):
    if WRITE_OUTPUTS:
        path = output_path(name)
        df.write.mode("overwrite").parquet(path)
        print(f"Saved {name} to {path}")

@contextmanager
def timer(stage_name: str):
    start = time.time()
    yield
    elapsed = time.time() - start
    runtime_results.append((stage_name, float(elapsed)))
    print(f"{stage_name}: {elapsed:.2f}s")

print("Spark version:", spark.version)
print("RAW_INPUT_PATH:", cfg.RAW_INPUT_PATH)
print("DEMO_MODE:", DEMO_MODE)
print("WRITE_OUTPUTS:", WRITE_OUTPUTS)



Spark version: 3.5.0
RAW_INPUT_PATH: abfss://2026-group4@sauondbrwebigdatalrs1.dfs.core.windows.net/taxi_log_2008_by_id/
DEMO_MODE: False
WRITE_OUTPUTS: False


%md
## Geospatial Utilities

Core functions for distance, direction, angle comparison, and spatial grid encoding used in trip segmentation and route similarity.

In [ ]:
EARTH_RADIUS_M = 6_371_000.0  # Earth radius in meters


# Haversine formula
def haversine_m(lon1: Column, lat1: Column, lon2: Column, lat2: Column) -> Column:
    lon1_rad = F.radians(lon1)
    lat1_rad = F.radians(lat1)
    lon2_rad = F.radians(lon2)
    lat2_rad = F.radians(lat2)
    dlon = lon2_rad - lon1_rad
    dlat = lat2_rad - lat1_rad
    a = F.pow(F.sin(dlat / 2.0), 2) + F.cos(lat1_rad) * F.cos(lat2_rad) * F.pow(F.sin(dlon / 2.0), 2)
    c = 2.0 * F.atan2(F.sqrt(a), F.sqrt(1.0 - a))
    return F.lit(EARTH_RADIUS_M) * c


# Bearing from point A to B 
def heading_rad(lon1: Column, lat1: Column, lon2: Column, lat2: Column) -> Column:
    lon1_rad = F.radians(lon1)
    lat1_rad = F.radians(lat1)
    lon2_rad = F.radians(lon2)
    lat2_rad = F.radians(lat2)
    dlon = lon2_rad - lon1_rad
    y = F.sin(dlon) * F.cos(lat2_rad)
    x = F.cos(lat1_rad) * F.sin(lat2_rad) - F.sin(lat1_rad) * F.cos(lat2_rad) * F.cos(dlon)
    return F.atan2(y, x)


# Smallest angular difference between two headings
def angle_diff_rad(a: Column, b: Column) -> Column:
    diff = F.abs(a - b)
    return F.least(diff, F.lit(2.0 * math.pi) - diff)


# Assign each point to a spatial grid cell 
def add_grid_cell(
    df: DataFrame,
    lon_col: str,
    lat_col: str,
    cell_size_deg: float,
    min_lon: float,
    min_lat: float,
    prefix: str = "",
) -> DataFrame:
    px = f"{prefix}cell_x" if prefix else "cell_x"
    py = f"{prefix}cell_y" if prefix else "cell_y"
    pid = f"{prefix}cell_id" if prefix else "cell_id"
    return (
        df.withColumn(px, F.floor((F.col(lon_col) - F.lit(min_lon)) / F.lit(cell_size_deg)).cast("long"))
          .withColumn(py, F.floor((F.col(lat_col) - F.lit(min_lat)) / F.lit(cell_size_deg)).cast("long"))
          .withColumn(pid, F.concat_ws("_", F.col(px), F.col(py)))
    )

%md
## GPS Cleaning & Validation

Transforms raw GPS data into a consistent schema, removes invalid points, and filters unrealistic movements using distance, time, and speed constraints.

In [ ]:

def normalize_raw_schema(raw_df: DataFrame) -> DataFrame:
    
    cols = raw_df.columns
    if {"taxi_id", "timestamp", "lon", "lat"}.issubset(set(cols)):
        df = raw_df.select("taxi_id", "timestamp", "lon", "lat")
    elif len(cols) >= 4:
        df = raw_df.select(
            F.col(cols[0]).alias("taxi_id"),
            F.col(cols[1]).alias("timestamp"),
            F.col(cols[2]).alias("lon"),
            F.col(cols[3]).alias("lat"),
        )
    else:
        raise ValueError("Expected at least four raw columns")

    return (
        df.withColumn("taxi_id", F.col("taxi_id").cast("string"))
          .withColumn("event_time", F.to_timestamp("timestamp"))
          .withColumn("lon", F.col("lon").cast("double"))
          .withColumn("lat", F.col("lat").cast("double"))
          .drop("timestamp")
    )


# Remove invalid, null, out-of-bound, and duplicate GPS points
def clean_gps_points(df: DataFrame, cfg: PipelineConfig) -> DataFrame:
    return (
        df.dropDuplicates(["taxi_id", "event_time", "lon", "lat"])
          .withColumn("is_null_record", F.col("taxi_id").isNull() | F.col("event_time").isNull() | F.col("lon").isNull() | F.col("lat").isNull())
          .withColumn("is_valid_lonlat", F.col("lon").between(-180, 180) & F.col("lat").between(-90, 90))
          .withColumn("is_zero_coordinate", (F.col("lon") == 0.0) & (F.col("lat") == 0.0))
          .withColumn("is_inside_beijing", F.col("lon").between(cfg.MIN_LON, cfg.MAX_LON) & F.col("lat").between(cfg.MIN_LAT, cfg.MAX_LAT))
          .withColumn("is_valid_basic", (~F.col("is_null_record")) & F.col("is_valid_lonlat") & (~F.col("is_zero_coordinate")) & F.col("is_inside_beijing"))
          .filter(F.col("is_valid_basic"))
          .select("taxi_id", "event_time", "lon", "lat")
    )


# Add movement-based validation (time, distance, speed)
def add_movement_quality(df: DataFrame, cfg: PipelineConfig) -> DataFrame:
    w = Window.partitionBy("taxi_id").orderBy("event_time")
    enriched = (
        df.withColumn("prev_time", F.lag("event_time").over(w))
          .withColumn("prev_lon", F.lag("lon").over(w))
          .withColumn("prev_lat", F.lag("lat").over(w))
          .withColumn("time_diff_sec", F.unix_timestamp("event_time") - F.unix_timestamp("prev_time"))

          # Distance + speed between consecutive points
          .withColumn("dist_from_prev_m", haversine_m(F.col("prev_lon"), F.col("prev_lat"), F.col("lon"), F.col("lat")))
          .withColumn("speed_from_prev_kmh", (F.col("dist_from_prev_m") / 1000.0) / (F.col("time_diff_sec") / 3600.0))

          # Detect unrealistic movement
          .withColumn("is_negative_or_zero_time", F.col("time_diff_sec").isNotNull() & (F.col("time_diff_sec") <= 0))
          .withColumn("is_impossible_speed", F.col("speed_from_prev_kmh").isNotNull() & (F.col("speed_from_prev_kmh") > cfg.MAX_SPEED_KMH))
          .withColumn("is_large_jump", F.col("dist_from_prev_m").isNotNull() & (F.col("dist_from_prev_m") > cfg.MAX_POINT_JUMP_KM * 1000.0))

          # Keep only valid transitions
          .withColumn("is_valid_movement", F.col("prev_time").isNull() | ((~F.col("is_negative_or_zero_time")) & (~F.col("is_impossible_speed"))))
    )
    return enriched.filter(F.col("is_valid_movement"))


# Simple record count summary across cleaning stages
def gps_quality_report(raw_df: DataFrame, clean_df: DataFrame, validated_df: DataFrame) -> DataFrame:
    spark = raw_df.sparkSession
    rows = [
        ("raw_records", raw_df.count()),
        ("after_basic_cleaning", clean_df.count()),
        ("after_movement_validation", validated_df.count()),
    ]
    return spark.createDataFrame(rows, ["metric", "value"])

%md
## Trip Construction

Builds trips from sequential GPS points using time, distance, and speed thresholds, then generates per trip summaries.

In [ ]:
# Split continuous GPS points into trips 
def construct_trips(points: DataFrame, cfg: PipelineConfig) -> DataFrame:
    w = Window.partitionBy("taxi_id").orderBy("event_time")
    w_cum = Window.partitionBy("taxi_id").orderBy("event_time").rowsBetween(Window.unboundedPreceding, 0)

    df = (
        points.withColumn("prev_time", F.lag("event_time").over(w))
              .withColumn("prev_lon", F.lag("lon").over(w))
              .withColumn("prev_lat", F.lag("lat").over(w))
              .withColumn("gap_minutes", (F.unix_timestamp("event_time") - F.unix_timestamp("prev_time")) / 60.0)

              # Movement metrics between consecutive points
              .withColumn("dist_from_prev_m", haversine_m(F.col("prev_lon"), F.col("prev_lat"), F.col("lon"), F.col("lat")))
              .withColumn("speed_from_prev_kmh", (F.col("dist_from_prev_m") / 1000.0) / (F.col("gap_minutes") / 60.0))

              .withColumn(
                  "new_trip_flag",
                  F.when(F.col("prev_time").isNull(), 1)
                   .when(F.col("gap_minutes") > cfg.TRIP_GAP_MINUTES, 1)
                   .when(F.col("dist_from_prev_m") > cfg.MAX_POINT_JUMP_KM * 1000.0, 1)
                   .when(F.col("speed_from_prev_kmh") > cfg.MAX_SPEED_KMH, 1)
                   .otherwise(0),
              )

              # Assign cumulative trip IDs per taxi
              .withColumn("trip_num", F.sum("new_trip_flag").over(w_cum))
              .withColumn("trip_id", F.concat_ws("_", F.col("taxi_id"), F.col("trip_num")))
              .withColumn("point_order", F.row_number().over(Window.partitionBy("trip_id").orderBy("event_time")))
    )

    # Keep only trips with at least 2 points
    trip_counts = df.groupBy("trip_id").agg(F.count("*").alias("point_count"))
    return (
        df.join(trip_counts, "trip_id")
          .filter(F.col("point_count") >= 2)
          .drop("prev_time", "prev_lon", "prev_lat")
    )


# Aggregate trip level statistics
def trip_summary(trips: DataFrame) -> DataFrame:
    return (
        trips.groupBy("trip_id", "taxi_id")
             .agg(
                 F.min("event_time").alias("trip_start_time"),
                 F.max("event_time").alias("trip_end_time"),
                 F.count("*").alias("point_count"),
                 F.min("lon").alias("min_lon"),
                 F.max("lon").alias("max_lon"),
                 F.min("lat").alias("min_lat"),
                 F.max("lat").alias("max_lat"),
             )
             .withColumn("trip_duration_sec", F.unix_timestamp("trip_end_time") - F.unix_timestamp("trip_start_time"))
    )

%md
## Segment Generation

Converts trip points into movement segments with distance, speed, direction, and motion dynamics.

In [ ]:
# Convert ordered trip points to movement segments 
def build_segments(trips: DataFrame) -> DataFrame:
    w = Window.partitionBy("trip_id").orderBy("event_time", "point_order")

    base = (
        trips.withColumn("next_time", F.lead("event_time").over(w))
             .withColumn("next_lon", F.lead("lon").over(w))
             .withColumn("next_lat", F.lead("lat").over(w))
             .withColumn("segment_idx", F.row_number().over(w))
             .filter(F.col("next_time").isNotNull())  # keep valid pairs only
    )

    seg = (
        base.withColumn("segment_id", F.concat_ws("_", F.col("trip_id"), F.col("segment_idx")))
            .withColumnRenamed("event_time", "start_time")
            .withColumnRenamed("lon", "start_lon")
            .withColumnRenamed("lat", "start_lat")
            .withColumnRenamed("next_time", "end_time")
            .withColumnRenamed("next_lon", "end_lon")
            .withColumnRenamed("next_lat", "end_lat")

            .withColumn("duration_sec", F.unix_timestamp("end_time") - F.unix_timestamp("start_time"))
            .withColumn("distance_m", haversine_m(F.col("start_lon"), F.col("start_lat"), F.col("end_lon"), F.col("end_lat")))
            .withColumn("speed_kmh", (F.col("distance_m") / 1000.0) / (F.col("duration_sec") / 3600.0))
            .withColumn("heading_rad", heading_rad(F.col("start_lon"), F.col("start_lat"), F.col("end_lon"), F.col("end_lat")))

            .withColumn("mid_lon", (F.col("start_lon") + F.col("end_lon")) / 2.0)
            .withColumn("mid_lat", (F.col("start_lat") + F.col("end_lat")) / 2.0)
            .withColumn("min_lon", F.least("start_lon", "end_lon"))
            .withColumn("max_lon", F.greatest("start_lon", "end_lon"))
            .withColumn("min_lat", F.least("start_lat", "end_lat"))
            .withColumn("max_lat", F.greatest("start_lat", "end_lat"))
    )

    w_seg = Window.partitionBy("trip_id").orderBy("segment_idx")
    seg = (
        seg.withColumn("prev_heading_rad", F.lag("heading_rad").over(w_seg))
           .withColumn("turn_angle_rad", angle_diff_rad(F.col("heading_rad"), F.col("prev_heading_rad")))
           .withColumn("prev_speed_kmh", F.lag("speed_kmh").over(w_seg))
           .withColumn("acceleration_kmh_per_s", (F.col("speed_kmh") - F.col("prev_speed_kmh")) / F.col("duration_sec"))
           .drop("prev_heading_rad", "prev_speed_kmh")
    )

    return seg.select(
        "taxi_id", "trip_id", "segment_id", "segment_idx",
        "start_time", "end_time", "start_lon", "start_lat", "end_lon", "end_lat",
        "mid_lon", "mid_lat", "min_lon", "max_lon", "min_lat", "max_lat",
        "duration_sec", "distance_m", "speed_kmh", "heading_rad", "turn_angle_rad", "acceleration_kmh_per_s",
    )

%md
## Spatial Partitioning

Implements Grid, Quadtree, and KDB tree partitioning to distribute segments spatially and evaluate data skew.

In [ ]:
from typing import List, Dict


# Uniform grid partitioning
def uniform_grid_partition(segments: DataFrame, cfg: PipelineConfig) -> DataFrame:
    bounds = segments.agg(
        F.min("mid_lon").alias("min_lon"), F.max("mid_lon").alias("max_lon"),
        F.min("mid_lat").alias("min_lat"), F.max("mid_lat").alias("max_lat"),
    ).collect()[0]

    min_lon, max_lon = bounds["min_lon"], bounds["max_lon"]
    min_lat, max_lat = bounds["min_lat"], bounds["max_lat"]

    lon_span = max(max_lon - min_lon, 1e-9)
    lat_span = max(max_lat - min_lat, 1e-9)

    return (
        segments.withColumn("grid_x", F.floor((F.col("mid_lon") - F.lit(min_lon)) / F.lit(lon_span) * F.lit(cfg.GRID_X)).cast("int"))
                .withColumn("grid_y", F.floor((F.col("mid_lat") - F.lit(min_lat)) / F.lit(lat_span) * F.lit(cfg.GRID_Y)).cast("int"))
                .withColumn("grid_x", F.least(F.greatest(F.col("grid_x"), F.lit(0)), F.lit(cfg.GRID_X - 1)))
                .withColumn("grid_y", F.least(F.greatest(F.col("grid_y"), F.lit(0)), F.lit(cfg.GRID_Y - 1)))
                .withColumn("partition_id", F.concat_ws("_", F.lit("grid"), F.col("grid_x"), F.col("grid_y")))
    )


# Counts points inside a bounding box
def _count_in_box(df: DataFrame, box: Dict[str, float]) -> int:
    return df.filter(
        (F.col("mid_lon") >= box["min_lon"]) & (F.col("mid_lon") <= box["max_lon"]) &
        (F.col("mid_lat") >= box["min_lat"]) & (F.col("mid_lat") <= box["max_lat"])
    ).count()


# Build Quadtree partitions (recursive 4-way spatial split)
def build_quadtree_leaves(segments: DataFrame, cfg: PipelineConfig) -> List[Dict[str, float]]:
    bounds = segments.agg(
        F.min("mid_lon").alias("min_lon"), F.max("mid_lon").alias("max_lon"),
        F.min("mid_lat").alias("min_lat"), F.max("mid_lat").alias("max_lat"),
    ).collect()[0].asDict()

    root = {**bounds, "depth": 0, "leaf_id": "qt_0"}
    leaves, queue = [], [root]

    while queue:
        box = queue.pop(0)
        cnt = _count_in_box(segments, box)

        # Stop splitting if small enough or max depth reached
        if cnt <= cfg.QT_MAX_POINTS or box["depth"] >= cfg.QT_MAX_DEPTH:
            box["count"] = cnt
            leaves.append(box)
            continue

        mid_lon = (box["min_lon"] + box["max_lon"]) / 2.0
        mid_lat = (box["min_lat"] + box["max_lat"]) / 2.0

        children = [
            {"min_lon": box["min_lon"], "max_lon": mid_lon, "min_lat": box["min_lat"], "max_lat": mid_lat},
            {"min_lon": mid_lon, "max_lon": box["max_lon"], "min_lat": box["min_lat"], "max_lat": mid_lat},
            {"min_lon": box["min_lon"], "max_lon": mid_lon, "min_lat": mid_lat, "max_lat": box["max_lat"]},
            {"min_lon": mid_lon, "max_lon": box["max_lon"], "min_lat": mid_lat, "max_lat": box["max_lat"]},
        ]

        for i, child in enumerate(children):
            child["depth"] = box["depth"] + 1
            child["leaf_id"] = f"{box['leaf_id']}_{i}"
            queue.append(child)

    return leaves


# Assign each segment to a partition leaf
def assign_leaves(segments: DataFrame, leaves: List[Dict[str, float]], prefix: str) -> DataFrame:
    spark = segments.sparkSession
    leaf_rows = [
        (l["leaf_id"], float(l["min_lon"]), float(l["max_lon"]),
         float(l["min_lat"]), float(l["max_lat"]),
         int(l["depth"]), int(l.get("count", -1)))
        for l in leaves
    ]

    leaf_df = spark.createDataFrame(
        leaf_rows,
        ["partition_id", "leaf_min_lon", "leaf_max_lon", "leaf_min_lat", "leaf_max_lat", "leaf_depth", "leaf_count"]
    )

    return segments.join(
        F.broadcast(leaf_df),
        (segments.mid_lon >= leaf_df.leaf_min_lon) & (segments.mid_lon <= leaf_df.leaf_max_lon) &
        (segments.mid_lat >= leaf_df.leaf_min_lat) & (segments.mid_lat <= leaf_df.leaf_max_lat),
        "left",
    ).withColumn("partition_method", F.lit(prefix))


def quadtree_partition(segments: DataFrame, cfg: PipelineConfig) -> DataFrame:
    leaves = build_quadtree_leaves(segments, cfg)
    return assign_leaves(segments, leaves, "quadtree")


# Build KDB-tree
def build_kdbtree_leaves(segments: DataFrame, cfg: PipelineConfig) -> List[Dict[str, float]]:
    bounds = segments.agg(
        F.min("mid_lon").alias("min_lon"), F.max("mid_lon").alias("max_lon"),
        F.min("mid_lat").alias("min_lat"), F.max("mid_lat").alias("max_lat"),
    ).collect()[0].asDict()

    leaves = []
    queue = [{**bounds, "depth": 0, "leaf_id": "kdb_0"}]

    while queue:
        box = queue.pop(0)

        current = segments.filter(
            (F.col("mid_lon") >= box["min_lon"]) & (F.col("mid_lon") <= box["max_lon"]) &
            (F.col("mid_lat") >= box["min_lat"]) & (F.col("mid_lat") <= box["max_lat"])
        )
        cnt = current.count()

        if cnt <= cfg.KDB_MAX_POINTS or box["depth"] >= cfg.KDB_MAX_DEPTH:
            box["count"] = cnt
            leaves.append(box)
            continue

        # Split along dominant axis using median
        lon_span = box["max_lon"] - box["min_lon"]
        lat_span = box["max_lat"] - box["min_lat"]
        axis = "mid_lon" if lon_span >= lat_span else "mid_lat"
        split = current.approxQuantile(axis, [0.5], 0.01)[0]

        if axis == "mid_lon":
            children = [
                {"min_lon": box["min_lon"], "max_lon": split, "min_lat": box["min_lat"], "max_lat": box["max_lat"]},
                {"min_lon": split, "max_lon": box["max_lon"], "min_lat": box["min_lat"], "max_lat": box["max_lat"]},
            ]
        else:
            children = [
                {"min_lon": box["min_lon"], "max_lon": box["max_lon"], "min_lat": box["min_lat"], "max_lat": split},
                {"min_lon": box["min_lon"], "max_lon": box["max_lon"], "min_lat": split, "max_lat": box["max_lat"]},
            ]

        for i, child in enumerate(children):
            child["depth"] = box["depth"] + 1
            child["leaf_id"] = f"{box['leaf_id']}_{i}"
            queue.append(child)

    return leaves


def kdbtree_partition(segments: DataFrame, cfg: PipelineConfig) -> DataFrame:
    leaves = build_kdbtree_leaves(segments, cfg)
    return assign_leaves(segments, leaves, "kdbtree")


# Compute partition balance and skew metrics
def partition_metrics(df: DataFrame, method_name: str) -> DataFrame:
    counts = df.groupBy("partition_id").agg(F.count("*").alias("partition_size"))

    metrics = counts.agg(
        F.count("*").alias("num_partitions"),
        F.min("partition_size").alias("min_partition_size"),
        F.max("partition_size").alias("max_partition_size"),
        F.avg("partition_size").alias("avg_partition_size"),
        F.stddev("partition_size").alias("std_partition_size"),
    ).withColumn("method", F.lit(method_name))

    return metrics.withColumn("skew_ratio", F.col("max_partition_size") / F.col("avg_partition_size"))

%md
## Trip Cell Sequences & Transitions

Encodes trips into spatial cell sequences, enriches with distance features, and extracts cell to cell transitions for route analysis.

In [ ]:
# Assign grid cells to each GPS point
def add_cells_to_trip_points(points: DataFrame, cfg: PipelineConfig) -> DataFrame:
    return add_grid_cell(points, "lon", "lat", cfg.CELL_SIZE_DEG, cfg.MIN_LON, cfg.MIN_LAT, prefix="")


# Build ordered cell sequences per trip
def build_trip_cell_sequences(trip_points: DataFrame, cfg: PipelineConfig) -> DataFrame:
    points = add_cells_to_trip_points(trip_points, cfg)

    seq = (
        points.groupBy("trip_id", "taxi_id")
              .agg(
                  F.sort_array(F.collect_list(F.struct("point_order", "event_time", "cell_id", "lon", "lat"))).alias("ordered_points"),
                  F.min("event_time").alias("trip_start_time"),
                  F.max("event_time").alias("trip_end_time"),
                  F.count("*").alias("point_count"),
              )

              # Extract cell sequence in order
              .withColumn("cell_sequence_raw", F.expr("transform(ordered_points, x -> x.cell_id)"))
              .withColumn(
                    "cell_sequence",
                    F.expr("""
                        aggregate(
                            cell_sequence_raw,
                            cast(array() as array<string>),
                            (acc, x) -> IF(
                                size(acc) = 0 OR element_at(acc, -1) <> x,
                                concat(acc, array(x)),
                                acc
                            )
                        )
                    """),
                )

              .withColumn("unique_cells", F.array_distinct("cell_sequence"))
              .withColumn("sequence_length", F.size("cell_sequence"))

              # Origin Destination representation
              .withColumn("origin_cell", F.element_at("cell_sequence", 1))
              .withColumn("destination_cell", F.element_at("cell_sequence", -1))
              .withColumn("od_pair", F.concat_ws("__", F.col("origin_cell"), F.col("destination_cell")))

              .withColumn("trip_duration_sec", F.unix_timestamp("trip_end_time") - F.unix_timestamp("trip_start_time"))
              .drop("ordered_points", "cell_sequence_raw")
    )

    return seq.filter(F.col("sequence_length") >= 2)


# Attach trip level distance and motion stats from segments
def add_trip_distance_from_segments(trip_sequences: DataFrame, segments: DataFrame) -> DataFrame:
    dist = segments.groupBy("trip_id").agg(
        F.sum("distance_m").alias("trip_distance_m"),
        F.avg("speed_kmh").alias("avg_speed_kmh"),
        F.max("speed_kmh").alias("max_speed_kmh"),
        F.avg("turn_angle_rad").alias("avg_turn_angle_rad"),
        F.count("*").alias("num_segments"),
    )
    return trip_sequences.join(dist, "trip_id", "left")


# Convert sequences to directed transitions
def build_route_transitions(trip_sequences: DataFrame) -> DataFrame:
    with_idx = trip_sequences.select(
        "trip_id", "taxi_id", "od_pair",
        F.posexplode("cell_sequence").alias("idx", "from_cell"),
        "cell_sequence",
    )

    return (
        with_idx.withColumn("to_cell", F.element_at("cell_sequence", F.col("idx") + 2))
                .filter(F.col("to_cell").isNotNull())
                .filter(F.col("from_cell") != F.col("to_cell"))
                .select("trip_id", "taxi_id", "od_pair", "idx", "from_cell", "to_cell")
    )

## Cell Sequences & Route Transitions

Encodes trips into grid cell sequences, adds trip-level metrics, and derives cell to cell transitions.

In [ ]:
def add_cells_to_trip_points(points: DataFrame, cfg: PipelineConfig) -> DataFrame:
    return add_grid_cell(points, "lon", "lat", cfg.CELL_SIZE_DEG, cfg.MIN_LON, cfg.MIN_LAT, prefix="")


# Build ordered cell sequences
def build_trip_cell_sequences(trip_points: DataFrame, cfg: PipelineConfig) -> DataFrame:
    points = add_cells_to_trip_points(trip_points, cfg)

    seq = (
        points.groupBy("trip_id", "taxi_id")
              .agg(
                  F.sort_array(F.collect_list(F.struct("point_order", "event_time", "cell_id", "lon", "lat"))).alias("ordered_points"),
                  F.min("event_time").alias("trip_start_time"),
                  F.max("event_time").alias("trip_end_time"),
                  F.count("*").alias("point_count"),
              )
              .withColumn("cell_sequence_raw", F.expr("transform(ordered_points, x -> x.cell_id)"))

              # Compress repeated consecutive cells
              .withColumn(
                    "cell_sequence",
                    F.expr("""
                        aggregate(
                            cell_sequence_raw,
                            cast(array() as array<string>),
                            (acc, x) -> IF(
                                size(acc) = 0 OR element_at(acc, -1) <> x,
                                concat(acc, array(x)),
                                acc
                            )
                        )
                    """),
                )

              .withColumn("sequence_length", F.size("cell_sequence"))
              .withColumn("origin_cell", F.element_at("cell_sequence", 1))
              .withColumn("destination_cell", F.element_at("cell_sequence", -1))
              .withColumn("od_pair", F.concat_ws("__", F.col("origin_cell"), F.col("destination_cell")))
              .withColumn("trip_duration_sec", F.unix_timestamp("trip_end_time") - F.unix_timestamp("trip_start_time"))
              .drop("ordered_points", "cell_sequence_raw")
    )

    return seq.filter(F.col("sequence_length") >= 2)


# Add distance to motion stats from segments
def add_trip_distance_from_segments(trip_sequences: DataFrame, segments: DataFrame) -> DataFrame:
    dist = segments.groupBy("trip_id").agg(
        F.sum("distance_m").alias("trip_distance_m"),
        F.avg("speed_kmh").alias("avg_speed_kmh"),
        F.max("speed_kmh").alias("max_speed_kmh"),
        F.avg("turn_angle_rad").alias("avg_turn_angle_rad"),
        F.count("*").alias("num_segments"),
    )
    return trip_sequences.join(dist, "trip_id", "left")


# Convert sequences to transitions 
def build_route_transitions(trip_sequences: DataFrame) -> DataFrame:
    with_idx = trip_sequences.select(
        "trip_id", "taxi_id", "od_pair",
        F.posexplode("cell_sequence").alias("idx", "from_cell"),
        "cell_sequence",
    )

    return (
        with_idx.withColumn("to_cell", F.element_at("cell_sequence", F.col("idx") + 2))
                .filter(F.col("to_cell").isNotNull())
                .filter(F.col("from_cell") != F.col("to_cell"))
                .select("trip_id", "taxi_id", "od_pair", "idx", "from_cell", "to_cell")
    )

## Candidate Pair Generation

Groups trips by OD, applies spatial blocking, and filters candidate pairs to avoid expensive all pairs comparisons.

In [ ]:
# OD-level aggregation 
def od_group_stats(trip_sequences: DataFrame) -> DataFrame:
    return (
        trip_sequences.groupBy("od_pair", "origin_cell", "destination_cell")
                      .agg(
                          F.count("*").alias("trip_count"),
                          F.avg("trip_distance_m").alias("avg_distance_m"),
                          F.avg("trip_duration_sec").alias("avg_duration_sec"),
                      )
    )


# Generate candidate trip pairs using OD and shared-cell blocking
def generate_candidate_trip_pairs(trip_sequences: DataFrame, cfg: PipelineConfig) -> DataFrame:
    base = trip_sequences.select(
        "trip_id", "taxi_id", "od_pair", "cell_sequence", "unique_cells",
        "sequence_length", "trip_distance_m", "trip_duration_sec", "avg_speed_kmh",
    ).filter(F.col("trip_distance_m").isNotNull())

    exploded = base.select(
        "trip_id", "od_pair", "trip_distance_m", "trip_duration_sec", "sequence_length",
        F.explode("unique_cells").alias("shared_block_cell"),
    )

    a, b = exploded.alias("a"), exploded.alias("b")

    raw_pairs = (
        a.join(
            b,
            (F.col("a.od_pair") == F.col("b.od_pair")) &
            (F.col("a.shared_block_cell") == F.col("b.shared_block_cell")) &
            (F.col("a.trip_id") < F.col("b.trip_id")),
            "inner",
        )
        .select(
            F.col("a.od_pair").alias("od_pair"),
            F.col("a.trip_id").alias("trip_id_1"),
            F.col("b.trip_id").alias("trip_id_2"),
            F.col("a.trip_distance_m").alias("distance_1"),
            F.col("b.trip_distance_m").alias("distance_2"),
            F.col("a.trip_duration_sec").alias("duration_1"),
            F.col("b.trip_duration_sec").alias("duration_2"),
            F.col("a.shared_block_cell"),
        )
    )

    # Aggregate shared cells and apply pruning rules
    grouped = (
        raw_pairs.groupBy("od_pair", "trip_id_1", "trip_id_2", "distance_1", "distance_2", "duration_1", "duration_2")
                 .agg(F.countDistinct("shared_block_cell").alias("shared_cell_count"))
                 .filter(F.col("shared_cell_count") >= cfg.MIN_SHARED_CELLS)
                 .withColumn("distance_ratio_diff", F.abs(F.col("distance_1") - F.col("distance_2")) / F.greatest(F.col("distance_1"), F.col("distance_2")))
                 .withColumn("duration_ratio_diff", F.abs(F.col("duration_1") - F.col("duration_2")) / F.greatest(F.col("duration_1"), F.col("duration_2")))
                 .filter(F.col("distance_ratio_diff") <= cfg.MAX_DISTANCE_RATIO_DIFF)
                 .filter(F.col("duration_ratio_diff") <= cfg.MAX_DURATION_RATIO_DIFF)
    )

    # Attach sequences for similarity computation
    seq1 = base.select(
        F.col("trip_id").alias("trip_id_1"),
        F.col("cell_sequence").alias("cell_sequence_1"),
        F.col("unique_cells").alias("unique_cells_1"),
    )
    seq2 = base.select(
        F.col("trip_id").alias("trip_id_2"),
        F.col("cell_sequence").alias("cell_sequence_2"),
        F.col("unique_cells").alias("unique_cells_2"),
    )

    return grouped.join(seq1, "trip_id_1").join(seq2, "trip_id_2")


def candidate_reduction_metrics(spark, total_trips: int, od_groups: DataFrame, candidate_pairs: DataFrame) -> DataFrame:
    total_possible = total_trips * (total_trips - 1) // 2

    od_pair_upper_bound = od_groups.agg(
        F.sum(F.col("trip_count") * (F.col("trip_count") - 1) / 2).alias("v")
    ).collect()[0]["v"] or 0

    final_pairs = candidate_pairs.count()

    return spark.createDataFrame([
        ("all_possible_pairs", int(total_possible)),
        ("after_od_grouping_upper_bound", int(od_pair_upper_bound)),
        ("after_spatial_blocking_and_pruning", int(final_pairs)),
    ], ["stage", "pair_count"])

## Route Similarity

Computes similarity between trip pairs using Jaccard (set), LCSS (sequence), and Edit Distance, then combines them into a final score.

In [ ]:
from typing import List


# Longest Common Subsequence
def _lcss_len(a: List[str], b: List[str]) -> int:
    if not a or not b:
        return 0
    m, n = len(a), len(b)
    prev = [0] * (n + 1)
    for i in range(1, m + 1):
        curr = [0] * (n + 1)
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                curr[j] = prev[j - 1] + 1
            else:
                curr[j] = max(prev[j], curr[j - 1])
        prev = curr
    return prev[n]


# Edit distance
def _edit_distance(a: List[str], b: List[str]) -> int:
    if a is None or b is None:
        return 10**9
    m, n = len(a), len(b)
    prev = list(range(n + 1))
    for i in range(1, m + 1):
        curr = [i] + [0] * n
        for j in range(1, n + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            curr[j] = min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + cost)
        prev = curr
    return prev[n]


# Register UDFs for Spark
lcss_len_udf = F.udf(_lcss_len, IntegerType())
edit_distance_udf = F.udf(_edit_distance, IntegerType())


# Compute combined route similarity score
def add_route_similarity(candidate_pairs: DataFrame, cfg: PipelineConfig) -> DataFrame:
    df = (
        candidate_pairs
            # Set-based similarity
            .withColumn("intersection_size", F.size(F.array_intersect("unique_cells_1", "unique_cells_2")))
            .withColumn("union_size", F.size(F.array_union("unique_cells_1", "unique_cells_2")))
            .withColumn("jaccard_similarity", F.col("intersection_size") / F.col("union_size"))
            .filter(F.col("jaccard_similarity") >= cfg.MIN_JACCARD_SIMILARITY)

            # Sequence-based similarity
            .withColumn("lcss_len", lcss_len_udf("cell_sequence_1", "cell_sequence_2"))
            .withColumn("max_seq_len", F.greatest(F.size("cell_sequence_1"), F.size("cell_sequence_2")))
            .withColumn("lcss_similarity", F.col("lcss_len") / F.col("max_seq_len"))

            .withColumn("edit_distance", edit_distance_udf("cell_sequence_1", "cell_sequence_2"))
            .withColumn("edit_similarity", 1.0 - (F.col("edit_distance") / F.col("max_seq_len")))

            # Weighted final score
            .withColumn(
                "final_route_similarity",
                0.40 * F.col("jaccard_similarity") +
                0.40 * F.col("lcss_similarity") +
                0.20 * F.col("edit_similarity"),
            )
            .filter(F.col("final_route_similarity") >= cfg.MIN_SEQUENCE_SIMILARITY)
    )
    return df

## Frequent Route Mining (N-grams)

Extracts common sub-routes (n-grams) from trip cell sequences and filters by support.

In [ ]:
# Mine frequent route patterns
def mine_ngram_routes(trip_sequences: DataFrame, n_values=(3, 4, 5), min_support: int = 20) -> DataFrame:
    spark = trip_sequences.sparkSession
    all_patterns = None

    for n in n_values:
        # Generate sliding window n-grams from sequences
        patterns = (
            trip_sequences.select(
                "trip_id", "od_pair",
                F.expr(f"""
                    IF(
                        size(cell_sequence) >= {n},
                        transform(sequence(1, size(cell_sequence)-{n}+1), i -> slice(cell_sequence, i, {n})),
                        array()
                    )
                """).alias("patterns")
            )
            .select("od_pair", F.explode("patterns").alias("cell_pattern"))
            .groupBy("od_pair", "cell_pattern")
            .agg(F.count("*").alias("support_count"))
            .withColumn("pattern_length", F.lit(n))
            .filter(F.col("support_count") >= min_support) 
        )

        all_patterns = patterns if all_patterns is None else all_patterns.unionByName(patterns)

    if all_patterns is None:
        return spark.createDataFrame([], "od_pair string, cell_pattern array<string>, support_count long, pattern_length int")
    return all_patterns.withColumn(
        "route_pattern_id",
        F.sha2(F.concat_ws("|", F.col("od_pair"), F.concat_ws(">", F.col("cell_pattern"))), 256)
    )

## Route Graph & Clustering

Builds route transition graphs, constructs trip similarity networks, and applies label propagation for clustering.

In [ ]:

def build_route_graph_edges(trip_sequences: DataFrame) -> DataFrame:
    transitions = build_route_transitions(trip_sequences)

    return (
        transitions.groupBy("from_cell", "to_cell")
                   .agg(
                       F.countDistinct("trip_id").alias("trip_count"),
                       F.count("*").alias("transition_count"),
                       F.countDistinct("od_pair").alias("od_pair_count"),
                   )
                   .withColumn("edge_id", F.sha2(F.concat_ws("->", "from_cell", "to_cell"), 256))
    )


# Build similarity graph between trips
def build_trip_similarity_graph(route_similarity: DataFrame, cfg: PipelineConfig) -> DataFrame:
    edges = (
        route_similarity
            .filter(F.col("final_route_similarity") >= cfg.ROUTE_EDGE_SIMILARITY_THRESHOLD)
            .select(
                F.col("trip_id_1").alias("src"),
                F.col("trip_id_2").alias("dst"),
                F.col("final_route_similarity").alias("weight"),
                "od_pair",
            )
    )

    # Make graph undirected
    rev = edges.select(F.col("dst").alias("src"), F.col("src").alias("dst"), "weight", "od_pair")
    return edges.unionByName(rev)


# Label propagation clustering
def label_propagation_clusters(edges: DataFrame, max_iter: int = 8) -> DataFrame:
    edges_cached = edges.persist(StorageLevel.MEMORY_AND_DISK)
    edges_cached.count()

    # Initialize each node with its own label
    vertices = (
        edges_cached.select(F.col("src").alias("trip_id"))
                    .union(edges_cached.select(F.col("dst").alias("trip_id")))
                    .distinct()
                    .withColumn("cluster_id", F.col("trip_id"))
                    .localCheckpoint(eager=True)
    )

    for _ in range(max_iter):
        # Propagate smallest neighbor label
        nbr_labels = (
            edges_cached
                .join(
                    vertices.select(
                        F.col("trip_id").alias("dst"),
                        F.col("cluster_id").alias("neighbor_cluster"),
                    ),
                    "dst",
                )
                .groupBy(F.col("src").alias("trip_id"))
                .agg(F.min("neighbor_cluster").alias("min_neighbor_cluster"))
        )

        vertices = (
            vertices.join(nbr_labels, "trip_id", "left")
                    .withColumn(
                        "cluster_id",
                        F.least(
                            F.col("cluster_id"),
                            F.coalesce(F.col("min_neighbor_cluster"), F.col("cluster_id")),
                        ),
                    )
                    .drop("min_neighbor_cluster")
                    .localCheckpoint(eager=True)
        )

    return vertices


# Summarise clusters with representative stats
def route_cluster_summary(clusters: DataFrame, trip_sequences: DataFrame) -> DataFrame:
    joined = clusters.join(trip_sequences, "trip_id", "left")

    return (
        joined.groupBy("cluster_id", "od_pair")
              .agg(
                  F.countDistinct("trip_id").alias("trip_count"),
                  F.avg("trip_distance_m").alias("avg_distance_m"),
                  F.avg("trip_duration_sec").alias("avg_duration_sec"),
                  F.avg("avg_speed_kmh").alias("avg_speed_kmh"),
                  F.first("cell_sequence", ignorenulls=True).alias("representative_sequence"),
              )
    )

## Route-based Anomaly Detection

Computes interpretable anomaly scores using route rarity, graph transitions, and deviation from OD norms.

In [ ]:
# Min-max normalization
def minmax_by_col(df: DataFrame, col_name: str, out_col: str) -> DataFrame:
    stats = df.agg(F.min(col_name).alias("min_v"), F.max(col_name).alias("max_v")).collect()[0]
    min_v = stats["min_v"] or 0.0
    max_v = stats["max_v"] or 1.0
    denom = max(max_v - min_v, 1e-9)
    return df.withColumn(out_col, (F.col(col_name) - F.lit(min_v)) / F.lit(denom))


# Compute rarity of edges in route graph
def build_graph_edge_rarity(trip_sequences: DataFrame, route_graph_edges: DataFrame) -> DataFrame:
    transitions = build_route_transitions(trip_sequences)

    max_count = route_graph_edges.agg(F.max("trip_count").alias("m")).collect()[0]["m"] or 1

    edge_scores = route_graph_edges.withColumn(
        "edge_rarity",
        1.0 - (F.log1p("trip_count") / F.log1p(F.lit(max_count)))
    )

    return (
        transitions.join(edge_scores.select("from_cell", "to_cell", "edge_rarity"), ["from_cell", "to_cell"], "left")
                   .fillna({"edge_rarity": 1.0})
                   .groupBy("trip_id")
                   .agg(F.avg("edge_rarity").alias("graph_edge_rarity_score"))
    )

def route_based_anomaly_scores(
    trip_sequences: DataFrame,
    clusters: DataFrame,
    cluster_summary: DataFrame,
    route_graph_edges: DataFrame,
) -> DataFrame:

    trip_cluster = trip_sequences.join(clusters, "trip_id", "left")
    cluster_sizes = cluster_summary.select("cluster_id", "od_pair", F.col("trip_count").alias("cluster_trip_count"))
    od_stats = (
        trip_sequences.groupBy("od_pair")
                      .agg(
                          F.count("*").alias("od_trip_count"),
                          F.expr("percentile_approx(trip_duration_sec, 0.5)").alias("od_median_duration_sec"),
                          F.expr("percentile_approx(trip_distance_m, 0.5)").alias("od_median_distance_m"),
                          F.avg("avg_speed_kmh").alias("od_avg_speed_kmh"),
                          F.stddev("avg_speed_kmh").alias("od_std_speed_kmh"),
                      )
    )

    graph_rarity = build_graph_edge_rarity(trip_sequences, route_graph_edges)

    scored = (
        trip_cluster.join(cluster_sizes, ["cluster_id", "od_pair"], "left")
                    .join(od_stats, "od_pair", "left")
                    .join(graph_rarity, "trip_id", "left")
                    .fillna({"cluster_trip_count": 1, "graph_edge_rarity_score": 1.0})

                    # Core anomaly signals
                    .withColumn("route_rarity_score", 1.0 - (F.log1p("cluster_trip_count") / F.log1p(F.greatest(F.col("od_trip_count"), F.lit(1)))))
                    .withColumn("duration_deviation_score", F.abs(F.col("trip_duration_sec") - F.col("od_median_duration_sec")) / F.greatest(F.col("od_median_duration_sec"), F.lit(1.0)))
                    .withColumn("distance_deviation_score", F.abs(F.col("trip_distance_m") - F.col("od_median_distance_m")) / F.greatest(F.col("od_median_distance_m"), F.lit(1.0)))
                    .withColumn("speed_z_score", F.abs(F.col("avg_speed_kmh") - F.col("od_avg_speed_kmh")) / F.greatest(F.col("od_std_speed_kmh"), F.lit(1.0)))
    )

    # Normalize deviation scores
    scored = minmax_by_col(scored, "duration_deviation_score", "duration_score")
    scored = minmax_by_col(scored, "distance_deviation_score", "distance_score")
    scored = minmax_by_col(scored, "speed_z_score", "speed_score")

    # Weighted final anomaly score
    scored = scored.withColumn(
        "route_based_anomaly_score",
        0.30 * F.col("route_rarity_score")
        + 0.25 * F.col("graph_edge_rarity_score")
        + 0.20 * F.col("duration_score")
        + 0.15 * F.col("distance_score")
        + 0.10 * F.col("speed_score"),
    )

    return scored.select(
        "trip_id", "taxi_id", "od_pair", "cluster_id",
        "route_rarity_score", "graph_edge_rarity_score",
        "duration_score", "distance_score", "speed_score",
        "route_based_anomaly_score",
    )

def add_anomaly_reason(df: DataFrame) -> DataFrame:
    return df.withColumn(
        "anomaly_reason",
        F.concat_ws(
            "; ",
            F.when(F.col("route_rarity_score") > 0.7, F.lit("rare route cluster for this OD pair")),
            F.when(F.col("graph_edge_rarity_score") > 0.7, F.lit("uses rare spatial transitions")),
            F.when(F.col("duration_score") > 0.7, F.lit("duration strongly deviates from OD normal")),
            F.when(F.col("distance_score") > 0.7, F.lit("distance strongly deviates from OD normal")),
            F.when(F.col("speed_score") > 0.7, F.lit("speed pattern is unusual")),
            F.when(F.col("dense_ae_score") > 0.7, F.lit("high Dense AE reconstruction error")),
            F.when(F.col("lstm_ae_score") > 0.7, F.lit("high LSTM AE reconstruction error")),
        ),
    )

## Evaluation Metrics

Tracks runtime, dataset sizes, and clustering quality.

In [ ]:
from contextlib import contextmanager
from typing import List, Tuple


# Convert recorded runtimes → DataFrame
@contextmanager
def make_runtime_df(spark: SparkSession, runtime_results: List[Tuple[str, float]]) -> DataFrame:
    return spark.createDataFrame(runtime_results, ["stage", "runtime_seconds"])


# Count rows for key datasets
def basic_count_metrics(spark: SparkSession, named_dfs: List[Tuple[str, DataFrame]]) -> DataFrame:
    rows = [(name, df.count()) for name, df in named_dfs]
    return spark.createDataFrame(rows, ["dataset", "row_count"])


# Compare similarity within clusters vs across clusters
def cluster_quality_metrics(route_similarity: DataFrame, clusters: DataFrame) -> DataFrame:
    spark = route_similarity.sparkSession

    sim_edges = route_similarity.select("trip_id_1", "trip_id_2", "final_route_similarity")

    c1 = clusters.select(F.col("trip_id").alias("trip_id_1"), F.col("cluster_id").alias("cluster_1"))
    c2 = clusters.select(F.col("trip_id").alias("trip_id_2"), F.col("cluster_id").alias("cluster_2"))

    joined = sim_edges.join(c1, "trip_id_1").join(c2, "trip_id_2")

    intra = joined.filter(F.col("cluster_1") == F.col("cluster_2")) \
                  .agg(F.avg("final_route_similarity")) \
                  .collect()[0][0]

    inter = joined.filter(F.col("cluster_1") != F.col("cluster_2")) \
                  .agg(F.avg("final_route_similarity")) \
                  .collect()[0][0]

    return spark.createDataFrame([
        ("avg_intra_similarity", float(intra or 0.0)),
        ("avg_inter_similarity", float(inter or 0.0)),
    ], ["metric", "value"])

# Pipeline execution




## Bronze Ingestion

Loads raw CSV data, normalises schema, and caches for downstream processing.

In [ ]:
with timer("01_bronze_ingestion"):
    # Read raw data from source
    raw_df = (
        spark.read.option("header", str(cfg.RAW_HAS_HEADER).lower())
                   .option("sep", cfg.RAW_SEPARATOR)
                   .csv(cfg.RAW_INPUT_PATH)
    )

    # Standardize schema
    bronze_df = normalize_raw_schema(raw_df).cache()
    bronze_count = bronze_df.count()

print("Bronze row count:", bronze_count)

display_df(bronze_df)
save_if_enabled(bronze_df, "bronze_raw")

01_bronze_ingestion: 92.76s
Bronze row count: 17662984


taxi_id,lon,lat,event_time
6275,116.36838,39.90484,2008-02-02T13:30:44Z
6275,116.36838,39.90484,2008-02-02T13:30:45Z
6275,116.36838,39.90484,2008-02-02T13:30:46Z
6275,116.36838,39.90484,2008-02-02T13:30:47Z
6275,116.36838,39.90484,2008-02-02T13:30:48Z
6275,116.36838,39.90484,2008-02-02T13:30:49Z
6275,116.36838,39.90484,2008-02-02T13:30:50Z
6275,116.36838,39.90484,2008-02-02T13:30:51Z
6275,116.36838,39.90484,2008-02-02T13:30:54Z
6275,116.36838,39.90484,2008-02-02T13:30:56Z


## GPS Cleaning & Validation

Filters invalid GPS points and removes unrealistic movements, with quality reporting.

In [ ]:
with timer("02_gps_cleaning_quality"):
    # Basic cleaning and movement validation
    clean_df = clean_gps_points(bronze_df, cfg).cache()
    validated_df = add_movement_quality(clean_df, cfg).cache()
    quality_report_df = gps_quality_report(bronze_df, clean_df, validated_df)

print("Clean row count:", clean_df.count())
print("Validated row count:", validated_df.count())

display_df(quality_report_df)
display_df(validated_df)

save_if_enabled(clean_df, "gps_clean")
save_if_enabled(validated_df, "gps_validated")
save_if_enabled(quality_report_df, "gps_quality_report")

02_gps_cleaning_quality: 60.40s
Clean row count: 16179985
Validated row count: 16102306


metric,value
raw_records,17662984
after_basic_cleaning,16179985
after_movement_validation,16102306


taxi_id,event_time,lon,lat,prev_time,prev_lon,prev_lat,time_diff_sec,dist_from_prev_m,speed_from_prev_kmh,is_negative_or_zero_time,is_impossible_speed,is_large_jump,is_valid_movement
10096,2008-02-02T13:34:48Z,116.39175,39.88507,null,null,null,null,null,null,false,false,false,true
10096,2008-02-02T13:38:38Z,116.39199,39.88499,2008-02-02T13:34:48Z,116.39175,39.88507,230,22.32633848616268,0.3494557328268942,false,false,false,true
10096,2008-02-02T13:39:50Z,116.39578,39.88561,2008-02-02T13:38:38Z,116.39199,39.88499,72,330.6419679121453,16.532098395607264,false,false,false,true
10096,2008-02-02T13:44:55Z,116.39582,39.88556,2008-02-02T13:39:50Z,116.39578,39.88561,305,6.523706895679822,0.07700113057195855,false,false,false,true
10096,2008-02-02T13:46:41Z,116.39582,39.88556,2008-02-02T13:44:55Z,116.39582,39.88556,106,0.0,0.0,false,false,false,true
10096,2008-02-02T13:49:57Z,116.39582,39.88556,2008-02-02T13:46:41Z,116.39582,39.88556,196,0.0,0.0,false,false,false,true
10096,2008-02-02T14:02:01Z,116.40561,39.89127,2008-02-02T13:49:57Z,116.39582,39.88556,724,1049.1963802198295,5.21699857567871,false,false,false,true
10096,2008-02-02T14:06:53Z,116.41834,39.8851,2008-02-02T14:02:01Z,116.40561,39.89127,292,1284.6589341411366,15.838260831877026,false,false,false,true
10096,2008-02-02T14:14:59Z,116.4332,39.88307,2008-02-02T14:06:53Z,116.41834,39.8851,486,1287.860748700693,9.539709249634763,false,false,false,true
10096,2008-02-02T14:17:10Z,116.44855,39.88313,2008-02-02T14:14:59Z,116.4332,39.88307,131,1309.769669448602,35.99367030545776,false,false,false,true


## Trip Construction

Builds trips from validated GPS points and generates trip-level summaries.

In [ ]:
with timer("03_trip_construction"):
    # Construct trips from cleaned GPS points
    trips_df = construct_trips(
        validated_df.select("taxi_id", "event_time", "lon", "lat"),
        cfg
    ).cache()

    # Generate summary per trip
    trip_summary_df = trip_summary(trips_df).cache()
    trip_count = trips_df.select("trip_id").distinct().count()

print("Trip count:", trip_count)

display_df(trip_summary_df)
display_df(trips_df)

save_if_enabled(trips_df, "trips")
save_if_enabled(trip_summary_df, "trip_summary")

03_trip_construction: 64.50s
Trip count: 568444


trip_id,taxi_id,trip_start_time,trip_end_time,point_count,min_lon,max_lon,min_lat,max_lat,trip_duration_sec
10000_6,10000,2008-02-02T18:46:48Z,2008-02-02T19:22:02Z,8,116.26794,116.27729,39.84575,39.94546,2114
10000_67,10000,2008-02-04T02:11:29Z,2008-02-04T05:42:07Z,43,116.35933,116.35967,39.93653,39.93702,12638
10003_99,10003,2008-02-06T01:44:04Z,2008-02-06T02:04:12Z,5,117.00032,117.00035,40.5576,40.55763,1208
10005_21,10005,2008-02-04T16:29:51Z,2008-02-04T20:01:05Z,52,116.28189,116.42181,39.87586,39.93365,12674
10007_82,10007,2008-02-05T23:44:34Z,2008-02-06T01:55:24Z,27,116.08429,116.08437,39.75474,39.75491,7850
10009_31,10009,2008-02-06T13:17:55Z,2008-02-06T13:19:11Z,2,116.50346,116.52053,40.00646,40.01774,76
10010_53,10010,2008-02-05T00:07:00Z,2008-02-05T01:12:25Z,14,116.45367,116.45585,39.88931,39.89743,3925
10010_55,10010,2008-02-05T02:32:56Z,2008-02-05T02:37:58Z,2,116.45363,116.45365,39.88941,39.88947,302
10015_64,10015,2008-02-07T00:43:19Z,2008-02-07T00:46:07Z,2,116.22916,116.22919,40.21166,40.21216,168
1001_36,1001,2008-02-07T00:15:48Z,2008-02-07T02:01:23Z,40,116.30031,116.51125,39.87038,39.98769,6335


trip_id,taxi_id,event_time,lon,lat,gap_minutes,dist_from_prev_m,speed_from_prev_kmh,new_trip_flag,trip_num,point_order,point_count
10000_6,10000,2008-02-02T18:46:48Z,116.27669,39.94533,10.066666666666666,2784.784047412305,16.598050613715724,1,6,1,8
10000_6,10000,2008-02-02T18:51:50Z,116.27232,39.94546,5.033333333333333,372.8155710183389,4.444159124721921,0,6,2,8
10000_6,10000,2008-02-02T18:56:52Z,116.26794,39.94184,5.033333333333333,549.0467906084566,6.544928629769681,0,6,3,8
10000_6,10000,2008-02-02T19:01:54Z,116.26896,39.91997,5.033333333333333,2433.3877704187125,29.007271435454854,0,6,4,8
10000_6,10000,2008-02-02T19:06:56Z,116.26797,39.8856,5.033333333333333,3822.702529050069,45.5686394191399,0,6,5,8
10000_6,10000,2008-02-02T19:11:58Z,116.27653,39.85633,5.033333333333333,3335.6514547391457,39.76273257304942,0,6,6,8
10000_6,10000,2008-02-02T19:17:00Z,116.27687,39.84575,5.033333333333333,1176.8003039759615,14.028083093753182,0,6,7,8
10000_6,10000,2008-02-02T19:22:02Z,116.27729,39.85817,5.033333333333333,1381.5063016680044,16.468287039751047,0,6,8,8
10000_67,10000,2008-02-04T02:11:29Z,116.35959,39.93679,10.066666666666666,6.522373683315917,0.038875074933671025,1,67,1,43
10000_67,10000,2008-02-04T02:16:31Z,116.35958,39.93679,5.033333333333333,0.8525905645592019,0.01016333123315605,0,67,2,43


## Segment Generation

Creates movement segments from trips and computes segment-level features.

In [ ]:
with timer("04_segment_generation"):
    # Generate segments from trips
    segments_df = build_segments(trips_df).cache()
    segment_count = segments_df.count()

print("Segment count:", segment_count)

display_df(segments_df)
save_if_enabled(segments_df, "segments")

04_segment_generation: 41.95s
Segment count: 15353966


taxi_id,trip_id,segment_id,segment_idx,start_time,end_time,start_lon,start_lat,end_lon,end_lat,mid_lon,mid_lat,min_lon,max_lon,min_lat,max_lat,duration_sec,distance_m,speed_kmh,heading_rad,turn_angle_rad,acceleration_kmh_per_s
10000,10000_6,10000_6_1,1,2008-02-02T18:46:48Z,2008-02-02T18:51:50Z,116.27669,39.94533,116.27232,39.94546,116.274505,39.945395,116.27232,116.27669,39.94533,39.94546,302,372.8155710183389,4.444159124721921,-1.5319886844907413,null,null
10000,10000_6,10000_6_2,2,2008-02-02T18:51:50Z,2008-02-02T18:56:52Z,116.27232,39.94546,116.26794,39.94184,116.27013,39.94365,116.26794,116.27232,39.94184,39.94546,302,549.0467906084566,6.544928629769681,-2.3936926635935993,0.861703979102858,0.006956190414065432
10000,10000_6,10000_6_3,3,2008-02-02T18:56:52Z,2008-02-02T19:01:54Z,116.26794,39.94184,116.26896,39.91997,116.26845,39.930904999999996,116.26794,116.26896,39.91997,39.94184,302,2433.3877704187125,29.007271435454854,3.105838325033682,0.7836543185523048,0.07437861856187143
10000,10000_6,10000_6_4,4,2008-02-02T19:01:54Z,2008-02-02T19:06:56Z,116.26896,39.91997,116.26797,39.8856,116.268465,39.902784999999994,116.26797,116.26896,39.8856,39.91997,302,3822.702529050069,45.5686394191399,-3.119494034037952,0.05785294810795172,0.05483896683339419
10000,10000_6,10000_6_5,5,2008-02-02T19:06:56Z,2008-02-02T19:11:58Z,116.26797,39.8856,116.27653,39.85633,116.27225,39.870965,116.26797,116.27653,39.85633,39.8856,302,3335.6514547391457,39.76273257304942,2.920751803694432,0.24293946944720268,-0.019224857106259867
10000,10000_6,10000_6_6,6,2008-02-02T19:11:58Z,2008-02-02T19:17:00Z,116.27653,39.85633,116.27687,39.84575,116.2767,39.85104,116.27653,116.27687,39.84575,39.85633,302,1176.8003039759615,14.028083093753182,3.116924449840269,0.19617264614583707,-0.08521407112349748
10000,10000_6,10000_6_7,7,2008-02-02T19:17:00Z,2008-02-02T19:22:02Z,116.27687,39.84575,116.27729,39.85817,116.27708,39.851960000000005,116.27687,116.27729,39.84575,39.85817,302,1381.5063016680044,16.468287039751047,0.025952782604380414,3.0909716672358885,0.008080145516549222
10000,10000_67,10000_67_1,1,2008-02-04T02:11:29Z,2008-02-04T02:16:31Z,116.35959,39.93679,116.35958,39.93679,116.359585,39.93679,116.35958,116.35959,39.93679,39.93679,302,0.8525905645592019,0.01016333123315605,-1.5707962707957916,null,null
10000,10000_67,10000_67_2,2,2008-02-04T02:16:31Z,2008-02-04T02:21:33Z,116.35958,39.93679,116.35965,39.93676,116.35961499999999,39.936775,116.35958,116.35965,39.93676,39.93679,302,6.837142600754432,0.08150236212819853,2.0804792831683683,2.6319097532154263,2.3622195660610095E-4
10000,10000_67,10000_67_3,3,2008-02-04T02:21:33Z,2008-02-04T02:26:35Z,116.35965,39.93676,116.35963,39.93675,116.35964,39.936755000000005,116.35963,116.35965,39.93675,39.93676,302,2.0357005726038007,0.024266629342296963,-2.148646328071706,2.054059695939512,-1.8952229399305155E-4


## Spatial Partitioning Evaluation

Applies Grid, Quadtree, and KDB-tree partitioning and compares their balance and skew.

In [ ]:
if RUN_PARTITIONING_EVALUATION:
    with timer("05_spatial_partitioning_evaluation"):
        partition_input_df = maybe_limit(segments_df).cache()
        partition_input_df.count()

        # Apply partitioning strategies
        grid_df = uniform_grid_partition(partition_input_df, cfg).cache()
        quadtree_df = quadtree_partition(partition_input_df, cfg).cache()
        kdb_df = kdbtree_partition(partition_input_df, cfg).cache()

        # Compute metrics for each method
        grid_metrics_df = partition_metrics(grid_df, "grid")
        quadtree_metrics_df = partition_metrics(quadtree_df, "quadtree")
        kdb_metrics_df = partition_metrics(kdb_df, "kdbtree")

        partition_metrics_df = (
            grid_metrics_df
            .unionByName(quadtree_metrics_df)
            .unionByName(kdb_metrics_df)
            .cache()
        )
        partition_metrics_df.count()
else:
    partition_metrics_df = spark.createDataFrame(
        [],
        "num_partitions long, min_partition_size long, max_partition_size long, avg_partition_size double, std_partition_size double, method string, skew_ratio double"
    )

print("Partition metrics:")
display_df(partition_metrics_df)

save_if_enabled(partition_metrics_df, "partition_metrics")

05_spatial_partitioning_evaluation: 528.27s
Partition metrics:


num_partitions,min_partition_size,max_partition_size,avg_partition_size,std_partition_size,method,skew_ratio
99,2,9716396,155090.56565656565,984310.1224983298,grid,62.649819857618546
112,385,1925459,137088.98214285713,305356.17724136374,quadtree,14.045322752440642
16,923502,995602,959746.625,18891.922502752343,kdbtree,1.037359209260048


## Cell Sequence Generation

Converts trips into cell sequences and enriches with trip-level distance features.

In [ ]:
with timer("06_cell_sequence_generation"):
    # Build cell sequences from trips
    trip_sequences_df = build_trip_cell_sequences(trips_df, cfg)

    # Add distance and motion stats
    trip_sequences_df = add_trip_distance_from_segments(
        trip_sequences_df, segments_df
    ).cache()

    route_sequence_count = trip_sequences_df.count()

print("Route sequence count:", route_sequence_count)

display_df(trip_sequences_df)
save_if_enabled(trip_sequences_df, "trip_cell_sequences")

06_cell_sequence_generation: 27.34s
Route sequence count: 480625


trip_id,taxi_id,trip_start_time,trip_end_time,point_count,cell_sequence,unique_cells,sequence_length,origin_cell,destination_cell,od_pair,trip_duration_sec,trip_distance_m,avg_speed_kmh,max_speed_kmh,avg_turn_angle_rad,num_segments
10000_6,10000,2008-02-02T18:46:48Z,2008-02-02T19:22:02Z,8,"List(255_189, 254_189, 253_188, 253_183, 253_177, 255_171, 255_169, 255_171)","List(255_189, 254_189, 253_188, 253_183, 253_177, 255_171, 255_169)",8,255_189,255_171,255_189__255_171,2114,13071.91072147869,22.260585902234286,45.5686394191399,0.8722158380986738,7
10005_21,10005,2008-02-04T16:29:51Z,2008-02-04T20:01:05Z,52,"List(275_182, 275_181, 275_180, 275_177, 280_178, 282_176, 283_176, 282_177, 281_177, 279_178, 275_177, 273_177, 272_177, 267_177, 263_178, 259_176, 256_175, 258_175, 260_176, 262_177, 263_176, 263_178, 262_177, 262_176, 262_175, 262_176, 263_177, 268_177, 271_177, 274_177, 275_177, 277_177, 278_178, 282_178, 283_179, 283_180, 284_180, 284_179, 281_179, 279_179, 278_181, 276_184, 275_184, 274_186)","List(275_182, 275_181, 275_180, 275_177, 280_178, 282_176, 283_176, 282_177, 281_177, 279_178, 273_177, 272_177, 267_177, 263_178, 259_176, 256_175, 258_175, 260_176, 262_177, 263_176, 262_176, 262_175, 263_177, 268_177, 271_177, 274_177, 277_177, 278_178, 282_178, 283_179, 283_180, 284_180, 284_179, 281_179, 279_179, 278_181, 276_184, 275_184, 274_186)",44,275_182,274_186,275_182__274_186,12674,43645.358190274244,12.432338895918365,26.971976930897434,0.7673019581285747,51
10009_31,10009,2008-02-06T13:17:55Z,2008-02-06T13:19:11Z,2,"List(304_203, 300_201)","List(304_203, 300_201)",2,304_203,300_201,304_203__300_201,76,1920.067786264088,90.95057934935153,90.95057934935153,null,1
10010_53,10010,2008-02-05T00:07:00Z,2008-02-05T01:12:25Z,14,"List(291_179, 290_177)","List(291_179, 290_177)",2,291_179,290_177,291_179__290_177,3925,960.2162343782445,0.8805005595263468,10.838031757383972,1.8499119964184174,13
1001_36,1001,2008-02-07T00:15:48Z,2008-02-07T02:01:23Z,40,"List(282_197, 279_197, 272_197, 266_196, 260_196, 261_196, 262_194, 266_193, 271_193, 272_190, 273_189, 273_186, 277_186, 280_186, 284_186, 285_182, 289_176, 291_174, 291_179, 291_184, 292_186, 291_186, 293_186, 296_186, 297_182, 299_182, 299_183, 299_185, 299_186, 298_186, 300_187, 302_188, 301_188)","List(282_197, 279_197, 272_197, 266_196, 260_196, 261_196, 262_194, 266_193, 271_193, 272_190, 273_189, 273_186, 277_186, 280_186, 284_186, 285_182, 289_176, 291_174, 291_179, 291_184, 292_186, 291_186, 293_186, 296_186, 297_182, 299_182, 299_183, 299_185, 299_186, 298_186, 300_187, 302_188, 301_188)",33,282_197,301_188,282_197__301_188,6335,49058.8787761021,27.924976613295794,76.08848888023259,0.9278177302513718,39
10021_71,10021,2008-02-06T12:03:03Z,2008-02-06T12:38:15Z,8,"List(279_182, 280_181)","List(279_182, 280_181)",2,279_182,280_181,279_182__280_181,2112,1138.7010669217304,1.940876004199273,9.898100070361977,0.7185641753407689,7
10028_8,10028,2008-02-04T02:14:29Z,2008-02-04T02:19:29Z,2,"List(268_174, 269_179)","List(268_174, 269_179)",2,268_174,269_179,268_174__269_179,300,2556.2268578897592,30.674722294677114,30.674722294677114,null,1
1002_33,1002,2008-02-04T06:36:25Z,2008-02-04T09:17:17Z,63,"List(284_187, 285_186, 287_178, 287_176, 283_172, 281_170, 281_169, 285_171, 286_171, 291_173, 292_174, 294_176, 291_174, 287_177, 287_179, 282_179, 281_179, 282_180, 282_179, 284_179, 284_178, 289_178, 290_178, 291_181, 289_182, 290_182, 291_188, 291_190, 294_195, 295_196, 297_198, 298_199, 301_202, 299_204, 297_204, 294_206, 294_207, 294_208, 296_208, 297_206, 297_208, 294_208, 297_207, 297_206, 297_204)","List(284_187, 285_186, 287_178, 287_176, 283_172, 281_170, 281_169, 285_171, 286_171, 291_173, 292_174, 294_176, 291_174, 287_177, 287_179, 282_179, 281_179, 282_180, 284_179, 284_178, 289_178, 290_178, 291_181, 289_182, 290_182, 291_188, 291_190, 294_195, 295_196, 297_198, 298_199, 301_202, 299_204, 297_204, 294_206, 294_207, 294_208, 296_208, 297_206, 297_208, 297_207)",45,

## Candidate Pair Generation

Generates candidate trip pairs using OD grouping and spatial blocking, with reduction analysis.

In [ ]:
with timer("07_candidate_pair_generation"):
    candidate_input_df = trip_sequences_df.localCheckpoint(eager=True)

    total_trips = candidate_input_df.count()

    # OD-level grouping
    od_stats_df = od_group_stats(candidate_input_df).localCheckpoint(eager=True)

    # Generate candidate pairs with pruning
    candidate_pairs_df = (
        generate_candidate_trip_pairs(candidate_input_df, cfg)
        .localCheckpoint(eager=True)
    )

    # Reduction metrics
    candidate_reduction_df = candidate_reduction_metrics(
        spark, total_trips, od_stats_df, candidate_pairs_df
    ).cache()

    candidate_pair_count = candidate_pairs_df.count()

print("Candidate pair count:", candidate_pair_count)

display_df(od_stats_df)
display_df(candidate_pairs_df)
display_df(candidate_reduction_df)

save_if_enabled(od_stats_df, "od_trip_groups")
save_if_enabled(candidate_pairs_df, "candidate_trip_pairs")
save_if_enabled(candidate_reduction_df, "candidate_reduction")

07_candidate_pair_generation: 50.72s
Candidate pair count: 207041


od_pair,origin_cell,destination_cell,trip_count,avg_distance_m,avg_duration_sec
291_179__290_177,291_179,290_177,5,2553.0507165483646,5322.2
289_191__293_193,289_191,293_193,4,5781.9327505662595,1294.25
289_190__262_193,289_190,262_193,2,31633.427712408535,5447.5
300_201__304_206,300_201,304_206,4,4090.0721613351616,4378.25
266_169__270_176,266_169,270_176,1,4440.755643591679,524.0
291_184__293_185,291_184,293_185,3,27338.778379060113,5086.0
270_181__269_186,270_181,269_186,7,3333.191723895405,924.5714285714286
255_155__215_130,255_155,215_130,1,23695.964936720597,6221.0
354_161__342_180,354_161,342_180,1,11635.517901343377,905.0
282_177__291_178,282_177,291_178,2,7067.62993343582,1701.0


trip_id_2,trip_id_1,od_pair,distance_1,distance_2,duration_1,duration_2,shared_cell_count,distance_ratio_diff,duration_ratio_diff,cell_sequence_1,unique_cells_1,cell_sequence_2,unique_cells_2
10018_4,10018_2,273_186__273_186,143.16621172365186,102.3042255735438,6299,3899,2,0.28541640976700805,0.38101285918399747,"List(273_186, 274_186, 273_186)","List(273_186, 274_186)","List(273_186, 274_186, 273_186)","List(273_186, 274_186)"
10050_37,10050_28,319_182__319_181,870.9619671394173,1204.7140238873458,302,302,2,0.27703840922427747,0.0,"List(319_182, 318_181, 319_181)","List(319_182, 318_181, 319_181)","List(319_182, 318_182, 319_181)","List(319_182, 318_182, 319_181)"
10056_27,10056_11,244_179__244_179,247.76458429171893,280.95502514009337,1798,2700,2,0.11813435560308842,0.3340740740740741,"List(244_179, 245_179, 244_179)","List(244_179, 245_179)","List(244_179, 245_179, 244_179, 245_179, 244_179)","List(244_179, 245_179)"
10061_49,10029_52,316_212__316_214,706.843021452372,1020.0855744502076,1510,1197,3,0.3070747796494063,0.20728476821192052,"List(316_212, 316_213, 316_214)","List(316_212, 316_213, 316_214)","List(316_212, 316_213, 316_214)","List(316_212, 316_213, 316_214)"
10078_88,10078_100,300_173__300_173,371.23098044161196,440.26625915773275,4501,7799,2,0.15680347353483598,0.4228747275291704,"List(300_173, 299_173, 300_173)","List(300_173, 299_173)","List(300_173, 299_173, 300_173)","List(300_173, 299_173)"
10078_88,10078_7,300_173__300_173,389.9981217103635,440.26625915773275,8399,7799,2,0.11417667468666926,0.07143707584236218,"List(300_173, 299_173, 300_173)","List(300_173, 299_173)","List(300_173, 299_173, 300_173)","List(300_173, 299_173)"
10078_88,10078_78,300_173__300_173,331.14578588876645,440.26625915773275,3301,7799,2,0.24785109237697014,0.576740607770227,"List(300_173, 299_173, 300_173)","List(300_173, 299_173)","List(300_173, 299_173, 300_173)","List(300_173, 299_173)"
10091_60,10091_29,325_244__327_246,2275.43558457625,2344.9541301772006,445,469,3,0.02964601512085742,0.0511727078891258,"List(325_244, 328_246, 327_246)","List(325_244, 328_246, 327_246)","List(325_244, 328_246, 327_246)","List(325_244, 328_246, 327_246)"
10112_60,10033_45,316_215__313_214,2690.746880748079,3114.088322751269,905,1262,2,0.135943941894741,0.28288431061806657,"List(316_215, 314_212, 313_214)","List(316_215, 314_212, 313_214)","List(316_215, 316_214, 312_212, 313_214)","List(316_215, 316_214, 312_212, 313_214)"
10146_35,10146_24,265_164__265_164,40471.72311328159,41036.89899485311,10071,6459,5,0.013772382792432902,0.35865355972594576,"List(265_164, 266_164, 267_164, 269_162, 267_163, 267_164, 270_167, 271_167, 271_168, 271_169, 271_172, 271_173, 267_173, 261_173, 260_171, 260_170, 260_171, 263_171, 263_173, 265_173, 269_173, 268_174, 270_175, 271_175, 269_177, 270_177, 269_176, 269_175, 268_174, 268_173, 276_174, 277_175, 278_175, 274_174, 273_170, 272_168, 272_167, 271_167, 268_166, 266_165, 265_164)","List(265_164, 266_164, 267_164, 269_162, 267_163, 270_167, 271_167, 271_168, 271_169, 271_172, 271_173, 267_173, 261_173, 260_171, 260_170, 263_171, 263_173, 265_173, 269_173, 268_174, 270_175, 271_175, 269_177, 270_177, 269_176, 269_175, 268_173, 276_174, 277_175, 278_175, 274_174, 273_170, 272_168, 272_167, 268_166, 266_165)","List(265_164, 267_165, 268_166, 266_169, 263_170, 260_176, 260_178, 262_179, 262_178, 263_179, 264_178, 263_176, 264_173, 261_173, 261_174, 256_175, 256_166, 259_166, 257_166, 256_166, 256_162, 257_161, 259_160, 259_158, 261_158, 261_159, 260_157, 261_158, 261_163, 261_164, 266_165, 267_164, 265_164)","List(265_164, 267_165, 268_166, 266_169, 263_170, 260_176, 260_178, 262_179, 262_178, 263_179, 264_178, 263_176, 264_173, 261_173, 261_174, 256_175, 256_166, 259_166, 257_166, 256_162, 257_161, 259_160, 259_158, 261_158, 261_159, 260_157, 261_163, 261_164, 266_165, 267_164)"


stage,pair_count
all_possible_pairs,115499955000
after_od_grouping_upper_bound,589491
after_spatial_blocking_and_pruning,207041


## Route Similarity & Mining

Computes route similarity between trips and extracts frequent route patterns (n-grams).

In [ ]:
with timer("08_route_similarity_and_mining"):
    # Compute similarity between candidate trip pairs
    route_similarity_df = add_route_similarity(candidate_pairs_df, cfg).cache()
    similar_pair_count = route_similarity_df.count()

    # Mine frequent route patterns
    frequent_routes_df = mine_ngram_routes(
        trip_sequences_df,
        n_values=(3, 4, 5),
        min_support=20
    ).cache()
    frequent_route_count = frequent_routes_df.count()

print("Similar route pair count:", similar_pair_count)
print("Frequent route pattern count:", frequent_route_count)

display_df(route_similarity_df)
display_df(frequent_routes_df)

save_if_enabled(route_similarity_df, "route_similarity")
save_if_enabled(frequent_routes_df, "frequent_routes")

08_route_similarity_and_mining: 62.55s
Similar route pair count: 191889
Frequent route pattern count: 892


trip_id_2,trip_id_1,od_pair,distance_1,distance_2,duration_1,duration_2,shared_cell_count,distance_ratio_diff,duration_ratio_diff,cell_sequence_1,unique_cells_1,cell_sequence_2,unique_cells_2,intersection_size,union_size,jaccard_similarity,lcss_len,max_seq_len,lcss_similarity,edit_distance,edit_similarity,final_route_similarity
10018_4,10018_2,273_186__273_186,143.16621172365186,102.3042255735438,6299,3899,2,0.28541640976700805,0.38101285918399747,"List(273_186, 274_186, 273_186)","List(273_186, 274_186)","List(273_186, 274_186, 273_186)","List(273_186, 274_186)",2,2,1.0,3,3,1.0,0,1.0,1.0
10050_37,10050_28,319_182__319_181,870.9619671394173,1204.7140238873458,302,302,2,0.27703840922427747,0.0,"List(319_182, 318_181, 319_181)","List(319_182, 318_181, 319_181)","List(319_182, 318_182, 319_181)","List(319_182, 318_182, 319_181)",2,4,0.5,2,3,0.6666666666666666,1,0.6666666666666667,0.6000000000000001
10056_27,10056_11,244_179__244_179,247.76458429171893,280.95502514009337,1798,2700,2,0.11813435560308842,0.3340740740740741,"List(244_179, 245_179, 244_179)","List(244_179, 245_179)","List(244_179, 245_179, 244_179, 245_179, 244_179)","List(244_179, 245_179)",2,2,1.0,3,5,0.6,2,0.6,0.76
10061_49,10029_52,316_212__316_214,706.843021452372,1020.0855744502076,1510,1197,3,0.3070747796494063,0.20728476821192052,"List(316_212, 316_213, 316_214)","List(316_212, 316_213, 316_214)","List(316_212, 316_213, 316_214)","List(316_212, 316_213, 316_214)",3,3,1.0,3,3,1.0,0,1.0,1.0
10078_88,10078_100,300_173__300_173,371.23098044161196,440.26625915773275,4501,7799,2,0.15680347353483598,0.4228747275291704,"List(300_173, 299_173, 300_173)","List(300_173, 299_173)","List(300_173, 299_173, 300_173)","List(300_173, 299_173)",2,2,1.0,3,3,1.0,0,1.0,1.0
10078_88,10078_7,300_173__300_173,389.9981217103635,440.26625915773275,8399,7799,2,0.11417667468666926,0.07143707584236218,"List(300_173, 299_173, 300_173)","List(300_173, 299_173)","List(300_173, 299_173, 300_173)","List(300_173, 299_173)",2,2,1.0,3,3,1.0,0,1.0,1.0
10078_88,10078_78,300_173__300_173,331.14578588876645,440.26625915773275,3301,7799,2,0.24785109237697014,0.576740607770227,"List(300_173, 299_173, 300_173)","List(300_173, 299_173)","List(300_173, 299_173, 300_173)","List(300_173, 299_173)",2,2,1.0,3,3,1.0,0,1.0,1.0
10091_60,10091_29,325_244__327_246,2275.43558457625,2344.9541301772006,445,469,3,0.02964601512085742,0.0511727078891258,"List(325_244, 328_246, 327_246)","List(325_244, 328_246, 327_246)","List(325_244, 328_246, 327_246)","List(325_244, 328_246, 327_246)",3,3,1.0,3,3,1.0,0,1.0,1.0
10112_60,10033_45,316_215__313_214,2690.746880748079,3114.088322751269,905,1262,2,0.135943941894741,0.28288431061806657,"List(316_215, 314_212, 313_214)","List(316_215, 314_212, 313_214)","List(316_215, 316_214, 312_212, 313_214)","List(316_215, 316_214, 312_212, 313_214)",2,5,0.4,2,4,0.5,2,0.5,0.4600000000000001
10222_11,10092_37,289_172__286_173,1903.9107254467676,3134.466970172468,911,852,2,0.39258867821407967,0.06476399560922064,"List(289_172, 289_171, 288_172, 286_173)","List(289_172, 289_171, 288_172, 286_173)","List(289_172, 291_173, 290_173, 286_173)","List(289_172, 291_173, 290_173, 286_173)",2,6,0.3333333333333333,2,4,0.5,2,0.5,0.43333333333333335


od_pair,cell_pattern,support_count,pattern_length,route_pattern_id
253_194__253_194,"List(253_194, 252_194, 253_194)",28,3,6c77e23af83a4bf94dfa017ce168d87fbd7076ff9607c042a85be84ee2f89ef5
312_214__317_215,"List(316_213, 316_214, 316_215)",58,3,302afeda597cfb4de00287aed50f7fdbb9b4e39d03b80c7bbb167a7f0d1c2722
263_149__263_149,"List(263_149, 263_150, 263_149)",25,3,9a5110a10d972c01d34bfabc5273ac2ee5d5a7b896821856f9dc1d57c3a7e99d
309_205__314_182,"List(314_182, 314_183, 314_182)",23,3,71b8a784041a6313b4e8372733ba1b544057267a40badbcceb69b9e4e9d7a03b
316_213__315_209,"List(316_213, 316_214, 316_215)",21,3,3f9cf0388be7a73c2b135da04d5f960a4f75fe34c682d0afc9de7b6ba56964f2
271_188__274_194,"List(272_188, 271_188, 272_188)",31,3,0f9f93c178a60152e849db7bfd98bbe94a7633593428513835aa054c1252a451
251_169__251_169,"List(251_170, 251_169, 251_170)",26,3,720c404231cfb30764f3ca507df686896c331a686b05f370499d9bec9ae1be08
268_202__268_202,"List(268_201, 268_202, 268_201)",57,3,8726fb5ed49790f80fbcd150e12de57d2d3e07db843187408696c1b6093001e0
312_214__315_209,"List(316_213, 316_214, 316_215)",32,3,8a757508daa1914b921db4b711caec9913542d4c077ec48cac908cc62bdd9954
313_214__315_209,"List(316_213, 316_214, 316_215)",45,3,1c80e349675696749224a81f7c2db748878e73cdf6903dab8e09dd609928abd2


## Route Graph Edges

Builds aggregated cell to cell transition edges from trip sequences.

In [ ]:
with timer("09a_build_route_graph_edges"):
    # Build route transition graph
    route_graph_edges_df = (
        build_route_graph_edges(trip_sequences_df)
        .repartition(64)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    route_graph_edge_count = route_graph_edges_df.count()

print("Route graph edge count:", route_graph_edge_count)

display(route_graph_edges_df.limit(100))
save_if_enabled(route_graph_edges_df, "route_graph_edges")

09a_build_route_graph_edges: 22.40s
Route graph edge count: 493381


from_cell,to_cell,trip_count,transition_count,od_pair_count,edge_id
278_189,275_190,27,27,27,5d0b19ac713bd4a7be4def4b19a1f7ee5354249e8d1b72d549549962600171ff
295_20,288_18,1,1,1,1be028694906a5044ea324eb9eb6d714dab99aa14a9052130d0f73f9b11f5b13
292_184,288_184,126,127,126,c424eb3c85653d27de6336c67abdc516fd8b64f7edcecb8bac13015ea12db8ae
264_185,257_186,2,2,2,cab3eba4ac2d56362774aae3a4047cb6b33b3c14b575bddf8f0adffdb6add3b6
261_206,264_213,1,1,1,8432d84a82fa6aca40eb873dc48ef4cb8f82d03ada266f67c5842d710620b3b3
255_201,257_198,3,3,3,ce53327babb93e8ae52470c6c9b1c2fc4f7efdaeef577ca20d242ffd6da09307
325_249,318_253,1,1,1,6b2cd381a54173da354b175b5ff80aa163134b0245dc8a1bc1f1af01b3ba85de
296_173,298_174,26,26,25,143aa3442536724182df41bdeeeb913fc2ef1ca52a8b50ef9219f3e396830c91
327_259,330_264,2,2,2,7165f5561aba0c312122cc4773da2518faaac52a71335cc80a36c01d407b0c4b
281_200,282_197,8,8,8,251ff0aa03d533e7d267d2664427b6c3f85106d5f88ed97445eb714cf0ce255e


%md
## Trip Similarity Graph

Constructs a graph where edges represent similar trips based on route similarity.

In [ ]:
with timer("09b_build_trip_similarity_graph"):
    # Build similarity graph between trips
    trip_graph_df = (
        build_trip_similarity_graph(route_similarity_df, cfg)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    trip_graph_edge_count = trip_graph_df.count()

print("Trip graph edge count:", trip_graph_edge_count)

display_df(trip_graph_df)
save_if_enabled(trip_graph_df, "trip_similarity_graph")

09b_build_trip_similarity_graph: 2.94s
Trip graph edge count: 336756


src,dst,weight,od_pair
10018_2,10018_4,1.0,273_186__273_186
10050_28,10050_37,0.6000000000000001,319_182__319_181
10056_11,10056_27,0.76,244_179__244_179
10029_52,10061_49,1.0,316_212__316_214
10078_100,10078_88,1.0,300_173__300_173
10078_7,10078_88,1.0,300_173__300_173
10078_78,10078_88,1.0,300_173__300_173
10091_29,10091_60,1.0,325_244__327_246
10033_45,10112_60,0.4600000000000001,316_215__313_214
10200_40,10257_3,1.0,269_188__269_186


%md
## Route Clustering

Applies label propagation on the trip similarity graph to identify route clusters.

In [ ]:
with timer("09c_label_propagation_clustering"):
    # Perform clustering using label propagation
    route_clusters_df = (
        label_propagation_clusters(trip_graph_df, max_iter=3)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    route_cluster_count = route_clusters_df.select("cluster_id").distinct().count()

print("Route cluster count:", route_cluster_count)

display_df(route_clusters_df)
save_if_enabled(route_clusters_df, "route_clusters")

09c_label_propagation_clustering: 43.96s
Route cluster count: 21785


trip_id,cluster_id
1526_16,1526_16
3004_104,3004_104
10226_41,10226_41
2462_28,10005_2
3717_17,1842_12
2822_125,2060_8
3073_42,3073_42
1333_61,10110_61
10294_25,10021_29
1943_18,1943_18


%md
## Cluster Summary

Generates aggregate statistics and representative routes for each cluster.

In [ ]:
with timer("09d_cluster_summary"):
    # Summarise clusters with key statistics
    cluster_summary_df = (
        route_cluster_summary(route_clusters_df, trip_sequences_df)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    cluster_summary_count = cluster_summary_df.count()

print("Cluster summary count:", cluster_summary_count)

display_df(cluster_summary_df)
save_if_enabled(cluster_summary_df, "route_cluster_summary")

09d_cluster_summary: 18.10s
Cluster summary count: 21785


cluster_id,od_pair,trip_count,avg_distance_m,avg_duration_sec,avg_speed_kmh,representative_sequence
10086_6,282_189__285_187,2,1976.4405585169993,331.0,22.290628750842433,"List(282_189, 285_187)"
10127_224,275_193__277_190,2,1637.5272949592122,302.0,19.52019292004359,"List(275_193, 277_190)"
10143_21,235_148__233_151,2,1914.3346249668307,964.0,6.871086567930732,"List(235_148, 233_151)"
10288_72,285_172__281_174,3,2038.7802789805821,748.0,9.319011146738973,"List(285_172, 281_174)"
106_13,274_179__273_179,4,421.0283828964194,188.0,9.133839793618941,"List(274_179, 273_179)"
1070_42,255_184__257_181,2,2339.172080814118,1423.5,6.603975624003063,"List(255_184, 257_184, 257_183, 257_181)"
1108_37,295_159__294_159,2,497.93991186699134,566.0,3.12876361137671,"List(295_159, 295_158, 294_158, 294_159)"
1118_23,298_197__300_201,2,2432.97429869697,270.5,35.61745081320496,"List(298_197, 299_200, 300_201)"
1122_28,296_197__289_191,2,4770.600539849609,301.0,57.05382979340239,"List(296_197, 289_191)"
1150_39,289_191__290_191,2,238.82287598800116,263.0,2.9673122446660205,"List(289_191, 290_191)"


%md
## Cluster Size Summary

Shows distribution of trips across clusters.

In [ ]:
with timer("09e_cluster_size_summary"):
    # Count trips per cluster
    cluster_size_summary_df = (
        route_clusters_df
        .groupBy("cluster_id")
        .count()
        .orderBy(F.desc("count"))
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    cluster_size_summary_count = cluster_size_summary_df.count()

print("Cluster size summary rows:", cluster_size_summary_count)

display_df(cluster_size_summary_df)

09e_cluster_size_summary: 5.72s
Cluster size summary rows: 21785


cluster_id,count
10002_78,234
10021_17,207
10005_2,195
10021_30,172
10027_64,134
1004_41,133
10002_44,131
10021_16,119
1003_12,109
10027_109,106


## Route Anomaly Scoring

Computes anomaly scores for trips and flags top anomalies based on a percentile threshold.

In [ ]:
with timer("10_route_anomaly_scoring"):
    # Compute route-based anomaly scores
    route_anomaly_scores_df = route_based_anomaly_scores(
        trip_sequences_df,
        route_clusters_df,
        cluster_summary_df,
        route_graph_edges_df
    ).cache()

    # Add placeholder AE scores and final score
    final_anomalies_df = (
        route_anomaly_scores_df
        .withColumn("dense_ae_score", F.lit(0.0))
        .withColumn("lstm_ae_score", F.lit(0.0))
        .withColumn("autoencoder_score", F.lit(0.0))
        .withColumn("final_anomaly_score", F.col("route_based_anomaly_score"))
    )

    final_anomalies_df = add_anomaly_reason(final_anomalies_df).cache()

    # Threshold = top 5% anomalies
    if final_anomalies_df.count() > 0:
        anomaly_threshold = final_anomalies_df.approxQuantile("final_anomaly_score", [0.95], 0.01)[0]
    else:
        anomaly_threshold = 1.0

    final_anomalies_df = final_anomalies_df.withColumn(
        "is_anomaly",
        F.col("final_anomaly_score") >= F.lit(anomaly_threshold)
    ).cache()

    anomaly_count = final_anomalies_df.filter(F.col("is_anomaly")).count()

print("Anomaly threshold:", anomaly_threshold)
print("Anomaly count:", anomaly_count)

display_df(final_anomalies_df.orderBy(F.desc("final_anomaly_score")))

save_if_enabled(route_anomaly_scores_df, "route_anomaly_scores")
save_if_enabled(final_anomalies_df, "anomalous_trips")

10_route_anomaly_scoring: 191.69s
Anomaly threshold: 0.30697841481711
Anomaly count: 28618


trip_id,taxi_id,od_pair,cluster_id,route_rarity_score,graph_edge_rarity_score,duration_score,distance_score,speed_score,route_based_anomaly_score,dense_ae_score,lstm_ae_score,autoencoder_score,final_anomaly_score,anomaly_reason,is_anomaly
7923_109,7923,300_201__300_201,null,0.8,0.8916847126725824,8.811977837817611E-5,5.850981174071276E-4,0.4708112108297186,0.5101076879244042,0.0,0.0,0.0,0.5101076879244042,rare route cluster for this OD pair; uses rare spatial transitions,true
7923_111,7923,300_201__300_201,null,0.8,0.916195854061945,8.792602336854953E-5,4.992596790231633E-4,0.23722060286990898,0.49286349795900436,0.0,0.0,0.0,0.49286349795900436,rare route cluster for this OD pair; uses rare spatial transitions,true
7219_5,7219,313_214__312_213,null,0.8522498868213139,0.6092121438863711,3.2123756106687044E-5,0.0012589285634851462,0.8207355865575441,0.49024682470948544,0.0,0.0,0.0,0.49024682470948544,rare route cluster for this OD pair; speed pattern is unusual,true
2364_2,2364,286_160__286_160,null,0.7373504649628064,0.916195854061945,6.661735995908762E-5,0.001716855625413415,0.3461811168177016,0.4851430665019022,0.0,0.0,0.0,0.4851430665019022,rare route cluster for this OD pair; uses rare spatial transitions,true
1019_24,1019,316_214__316_215,null,0.8735280338195903,0.7997101862714064,3.665118950674996E-5,7.939673186279823E-4,0.14880952737048608,0.4769933347864728,0.0,0.0,0.0,0.4769933347864728,rare route cluster for this OD pair; uses rare spatial transitions,true
7443_33,7443,301_201__303_203,null,0.772329751303047,0.7951998718343478,0.017203466080275374,0.00246369404425679,0.37021067417251813,0.4713302080894465,0.0,0.0,0.0,0.4713302080894465,rare route cluster for this OD pair; uses rare spatial transitions,true
5232_130,5232,262_179__263_179,null,0.814550976584631,0.8076725646979368,0.0014658842032065406,0.01038810217296233,0.2167309367909796,0.4698079199955571,0.0,0.0,0.0,0.4698079199955571,rare route cluster for this OD pair; uses rare spatial transitions,true
3385_23,3385,275_176__275_176,null,0.7757561757824246,0.916195854061945,7.433833040170684E-5,2.3092720035995282E-4,0.04941640562677212,0.46676696355902514,0.0,0.0,0.0,0.46676696355902514,rare route cluster for this OD pair; uses rare spatial transitions,true
4566_107,4566,313_214__313_207,null,0.7818957080144684,0.916195854061945,7.291114733877759E-5,9.06786583763448E-5,0.021893268452856628,0.46583518679333663,0.0,0.0,0.0,0.46583518679333663,rare route cluster for this OD pair; uses rare spatial transitions,true
1693_5,1693,300_202__300_202,null,0.7818957080144684,0.7895595323404858,1.7554203872169646E-5,0.0045801578894051,0.32277978321459083,0.46492710833510625,0.0,0.0,0.0,0.46492710833510625,rare route cluster for this OD pair; uses rare spatial transitions,true


## Evaluation & Metrics

Generates dataset counts, clustering quality, popular routes, and runtime performance metrics.

In [ ]:
with timer("11_evaluation_tables"):
    # Dataset size metrics
    count_metrics_df = basic_count_metrics(spark, [
        ("bronze_raw", bronze_df),
        ("gps_clean", clean_df),
        ("gps_validated", validated_df),
        ("trips", trips_df.select("trip_id").distinct()),
        ("segments", segments_df),
        ("trip_cell_sequences", trip_sequences_df),
        ("candidate_trip_pairs", candidate_pairs_df),
        ("route_similarity", route_similarity_df),
        ("route_clusters", route_clusters_df),
        ("anomalous_trips", final_anomalies_df.filter(F.col("is_anomaly"))),
    ]).cache()

    # Clustering quality
    cluster_quality_df = cluster_quality_metrics(route_similarity_df, route_clusters_df).cache()

    # OD-level patterns
    od_patterns_df = (
        trip_sequences_df.groupBy("od_pair", "origin_cell", "destination_cell")
            .agg(
                F.count("*").alias("trip_count"),
                F.avg("trip_distance_m").alias("avg_distance_m"),
                F.avg("trip_duration_sec").alias("avg_duration_sec"),
                F.avg("avg_speed_kmh").alias("avg_speed_kmh")
            )
            .orderBy(F.desc("trip_count"))
            .cache()
    )

    # Frequent route patterns
    popular_routes_df = frequent_routes_df.orderBy(F.desc("support_count")).cache()

    # Runtime tracking
    runtime_metrics_df = spark.createDataFrame(
        runtime_results, ["stage", "elapsed_sec"]
    ).cache()


print("Count metrics:")
display_df(count_metrics_df)

print("Partition metrics:")
display_df(partition_metrics_df)

print("Candidate reduction:")
display_df(candidate_reduction_df)

print("Cluster quality:")
display_df(cluster_quality_df)

print("Popular routes:")
display_df(popular_routes_df)

print("Runtime metrics:")
display_df(runtime_metrics_df)

save_if_enabled(count_metrics_df, "basic_count_metrics")
save_if_enabled(cluster_quality_df, "cluster_quality")
save_if_enabled(od_patterns_df, "od_patterns")
save_if_enabled(popular_routes_df, "popular_routes")
save_if_enabled(runtime_metrics_df, "runtime_metrics")

11_evaluation_tables: 111.15s
Count metrics:


dataset,row_count
bronze_raw,17662984
gps_clean,16179985
gps_validated,16102306
trips,568444
segments,15353966
trip_cell_sequences,480625
candidate_trip_pairs,207041
route_similarity,191889
route_clusters,64418
anomalous_trips,28618


Partition metrics:


num_partitions,min_partition_size,max_partition_size,avg_partition_size,std_partition_size,method,skew_ratio
99,2,9716396,155090.56565656565,984310.1224983298,grid,62.649819857618546
112,385,1925459,137088.98214285713,305356.17724136374,quadtree,14.045322752440642
16,923502,995602,959746.625,18891.922502752343,kdbtree,1.037359209260048


Candidate reduction:


stage,pair_count
all_possible_pairs,115499955000
after_od_grouping_upper_bound,589491
after_spatial_blocking_and_pruning,207041


... (2 outputs left out)

Popular routes:


od_pair,cell_pattern,support_count,pattern_length,route_pattern_id
313_214__316_215,"List(316_213, 316_214, 316_215)",139,3,b179a200c3c6b34e7a6fd247fd1a5af510e41a9a444393da60634baff7807436
313_214__316_215,"List(316_212, 316_213, 316_214)",115,3,54054e070bcbf2c6efdbacec9a8abd03aaded17aeec26b18e7526e14e5d2bd53
313_214__316_215,"List(316_212, 316_213, 316_214, 316_215)",114,4,edef28f0af3e9579cf276b1f7cf706b2a1137191ce774a2af042b7c53b16744a
267_181__267_181,"List(267_181, 267_182, 267_181)",107,3,90d31a32398fadbc7c0dd008dc33baef467d21ffee132b1b2687ec011e60545f
313_214__317_215,"List(316_212, 316_213, 316_214)",106,3,7ecb664df770e467fbf8bc300b45700be9bb03e99a81b8847ca4063820ed7873
316_213__316_215,"List(316_213, 316_214, 316_215)",101,3,477538c6ea1e6a69e0992a7f94a2e8ac7ed4295d0005c653c83c280d4abf8914
313_214__317_215,"List(316_213, 316_214, 316_215)",96,3,c32fc53cf7f77f0ba7f53efdbc5c06f5e90182e8b9f244defb4ab0c7ff522105
313_214__317_215,"List(316_214, 316_215, 317_215)",96,3,4df1de9955709e5fe4f3e65bc4fcddd8748e6682a515d9ed8e90973d5a0daad2
267_182__267_181,"List(267_182, 267_181, 267_182)",95,3,80201af57d357bcca53c6f109a008798e4d4ae0ae1af54ab8631bb94e31bb90b
267_182__267_181,"List(267_181, 267_182, 267_181)",94,3,1c12a1aa513a9312058079c440260314736f3adff02968bd96720172bae3dc89


Runtime metrics:


stage,elapsed_sec
01_bronze_ingestion,92.75777506828308
02_gps_cleaning_quality,60.401026010513306
03_trip_construction,64.5000901222229
04_segment_generation,41.95256066322327
05_spatial_partitioning_evaluation,528.2650420665741
06_cell_sequence_generation,27.34329080581665
07_candidate_pair_generation,50.72393250465393
08_route_similarity_and_mining,62.5463182926178
09a_build_route_graph_edges,22.404850244522095
09b_build_trip_similarity_graph,2.936481237411499


## Scalability Benchmark

Evaluates performance of key stages across increasing dataset sizes.

In [ ]:
benchmark_rows = []

def timed_benchmark_stage(dataset_name: str, stage_name: str, func, force_count: bool = True):
    start = time.time()
    result = func()

    row_count = None
    if force_count and result is not None:
        result = result.persist(StorageLevel.MEMORY_AND_DISK)
        row_count = result.count()

    runtime = time.time() - start

    benchmark_rows.append({
        "dataset": dataset_name,
        "stage": stage_name,
        "runtime_seconds": float(runtime),
        "row_count": int(row_count) if row_count is not None else None
    })

    print(f"{dataset_name} | {stage_name}: {runtime:.2f}s | rows={row_count}")
    return result


if RUN_SCALABILITY_BENCHMARK:
    # Base dataset
    seq_full = trip_sequences_df.persist(StorageLevel.MEMORY_AND_DISK)
    seq_full.count()

    # Different input sizes
    benchmark_datasets = {
        "small": seq_full.limit(10_000),
        "medium": seq_full.limit(50_000),
        "large": seq_full.limit(100_000)
    }

    for dataset_name, seq_df in benchmark_datasets.items():
        print(f"Running benchmark for {dataset_name}")

        seq_df = seq_df.persist(StorageLevel.MEMORY_AND_DISK)
        seq_count = seq_df.count()

        benchmark_rows.append({
            "dataset": dataset_name,
            "stage": "input_size",
            "runtime_seconds": 0.0,
            "row_count": int(seq_count)
        })

        # Benchmark key pipeline stages
        bench_candidates = timed_benchmark_stage(
            dataset_name, "candidate_pair_generation",
            lambda: generate_candidate_trip_pairs(seq_df, cfg)
        )

        bench_similarity = timed_benchmark_stage(
            dataset_name, "route_similarity",
            lambda: add_route_similarity(bench_candidates, cfg)
        )

        bench_graph = timed_benchmark_stage(
            dataset_name, "similarity_graph_construction",
            lambda: build_trip_similarity_graph(bench_similarity, cfg)
        )

        _bench_clusters = timed_benchmark_stage(
            dataset_name, "label_propagation_clustering",
            lambda: label_propagation_clusters(bench_graph, max_iter=8)
        )

    scalability_benchmark_df = spark.createDataFrame(benchmark_rows).cache()

    # Summary metrics
    scalability_benchmark_summary_df = (
        scalability_benchmark_df
            .filter(F.col("stage") != "input_size")
            .groupBy("stage")
            .agg(
                F.avg("runtime_seconds").alias("avg_runtime_seconds"),
                F.max("runtime_seconds").alias("max_runtime_seconds")
            )
            .orderBy(F.desc("max_runtime_seconds"))
            .cache()
    )

else:
    scalability_benchmark_df = spark.createDataFrame(
        [], "dataset string, stage string, runtime_seconds double, row_count long"
    )

    scalability_benchmark_summary_df = spark.createDataFrame(
        [], "stage string, avg_runtime_seconds double, max_runtime_seconds double"
    )


print("Scalability benchmark:")
display_df(scalability_benchmark_df)

print("Scalability benchmark summary:")
display_df(scalability_benchmark_summary_df)

save_if_enabled(scalability_benchmark_df, "scalability_benchmark")
save_if_enabled(scalability_benchmark_summary_df, "scalability_benchmark_summary")

===== Running benchmark for small =====
small | candidate_pair_generation: 9.01s | rows=76
small | route_similarity: 7.65s | rows=71
small | similarity_graph_construction: 45.63s | rows=132
small | label_propagation_clustering: 150.94s | rows=97
===== Running benchmark for medium =====
medium | candidate_pair_generation: 13.32s | rows=2341
medium | route_similarity: 6.60s | rows=2154
medium | similarity_graph_construction: 49.19s | rows=3852
medium | label_propagation_clustering: 100.49s | rows=1650
===== Running benchmark for large =====
large | candidate_pair_generation: 26.36s | rows=8931
large | route_similarity: 8.44s | rows=8246
large | similarity_graph_construction: 8.77s | rows=14714
large | label_propagation_clustering: 118.24s | rows=5300
Scalability benchmark:


dataset,row_count,runtime_seconds,stage
small,10000,0.0,input_size
small,76,9.010557174682617,candidate_pair_generation
small,71,7.645455360412598,route_similarity
small,132,45.63021397590637,similarity_graph_construction
small,97,150.94198346138,label_propagation_clustering
medium,50000,0.0,input_size
medium,2341,13.316635370254517,candidate_pair_generation
medium,2154,6.602675676345825,route_similarity
medium,3852,49.19194507598877,similarity_graph_construction
medium,1650,100.48552751541138,label_propagation_clustering


Scalability benchmark summary:


stage,avg_runtime_seconds,max_runtime_seconds
label_propagation_clustering,123.22083735466003,150.94198346138
similarity_graph_construction,34.53017155329386,49.19194507598877
candidate_pair_generation,16.230062166849773,26.362993955612183
route_similarity,7.5631271203358965,8.441250324249268
